In [ ]:
#@title Lesson 6 of 26: In Practice: Building a Classifier That Holds Up
from IPython.display import HTML, display
display(HTML('<div style="background:linear-gradient(118deg, #0d0726 0%, #2a0b5e 38%, #8f0f6b 72%, #ff6a1a 100%);border-radius:22px;padding:38px 36px 30px;color:#fff;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;box-shadow:0 12px 40px rgba(60,10,90,.35)"><div style="font-size:12px;letter-spacing:.34em;font-weight:700;color:#ffd1ea">BASIC PRINCIPLES OF ARTIFICIAL INTELLIGENCE  ·  LESSON 6 OF 26</div><div style="font-size:42px;font-weight:800;line-height:1.08;margin:14px 0 14px;text-shadow:0 2px 18px rgba(255,80,160,.45)">In Practice: Building a Classifier That Holds Up</div><div style="font-size:16px;line-height:1.55;max-width:680px;color:#f5ecff"></div><div style="margin-top:22px"><a href="https://youtu.be/" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#fff;color:#4a0d66">▶ Watch the lesson</a><a href="https://www.youtube.com/playlist?list=PLDyCOTShSfCk" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:rgba(255,255,255,.16);color:#fff;border:1px solid rgba(255,255,255,.45)">All lessons</a><a href="https://colab.research.google.com/github/neural-modules/ai-principles-notebooks/blob/main/06_building_a_classifier_that_holds_up.ipynb" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#ffb22e;color:#1a0d00">Open in Colab</a></div></div><div style="background:#141033;color:#ece8ff;border-radius:18px;padding:22px 26px;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;font-size:15px;line-height:1.5;border:1px solid #2c2466"><div style="font-size:12px;letter-spacing:.3em;font-weight:700;color:#a87bff;margin-bottom:6px">HOW THIS NOTEBOOK WORKS</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ff3d9a">1</span>Choose <b>File &gt; Save a copy in Drive</b> first, so your work is saved to your own account.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#a87bff">2</span>Run the set-up cell, then work down the page in order. Later exercises reuse earlier answers.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#2ad4ff">3</span>Replace each <code>...</code> with your answer and run the cell. The check runs straight away.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ffb22e">4</span>Stuck? Open the 💡 hint under the exercise. Some exercises have two.</div><div style="margin-top:14px;font-size:14px"><span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#11c48a;color:#03251a;padding:3px 10px;border-radius:999px;font-weight:700">✅ correct</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ff4d6d;color:#fff;padding:3px 10px;border-radius:999px;font-weight:700">❌ not yet, with a pointer</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ffb22e;color:#2a1700;padding:3px 10px;border-radius:999px;font-weight:700">⏳ still blank</span></div><div style="margin-top:16px;font-size:12px;color:#9d95c9">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</div></div>'))

In [ ]:
#@title Set-up: run this cell first
import base64, contextlib, html as _html, math, sys, zlib
import numpy as np

try:
    import torch
except ImportError:
    torch = None
try:
    import einops
except ImportError:
    get_ipython().run_line_magic("pip", "install -q einops")
    import einops


from IPython.display import HTML, display
import matplotlib.pyplot as plt
from cycler import cycler

PALETTE = {"pink": "#ff3d9a", "cyan": "#2ad4ff", "amber": "#ffb22e", "violet": "#a87bff",
           "mint": "#3dffb4", "orange": "#ff7a3d", "ink": "#ece8ff", "panel": "#141033", "night": "#0c0922"}
plt.rcParams.update({
    "figure.figsize": (6.8, 3.8), "figure.facecolor": "#0c0922", "axes.facecolor": "#141033",
    "savefig.facecolor": "#0c0922", "axes.edgecolor": "#3a3170", "axes.labelcolor": "#ece8ff",
    "axes.titlecolor": "#ffffff", "axes.titleweight": "bold", "axes.titlesize": 12,
    "text.color": "#ece8ff", "xtick.color": "#b9b2e6", "ytick.color": "#b9b2e6",
    "axes.grid": True, "grid.color": "#ffffff", "grid.alpha": 0.08, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "lines.linewidth": 2.4, "image.cmap": "plasma",
    "axes.prop_cycle": cycler(color=["#2ad4ff", "#ff3d9a", "#ffb22e", "#a87bff", "#3dffb4", "#ff7a3d"]),
})

_CARD = {"pass": ("linear-gradient(90deg,#0fae7a,#3dffb4)", "#03251a"),
         "wrong": ("linear-gradient(90deg,#e0245e,#ff6b6b)", "#ffffff"),
         "todo": ("linear-gradient(90deg,#ff9f1c,#ffd166)", "#2a1700")}

def _say(state: str, text: str):
    bg, fg = _CARD[state]
    try:
        display(HTML(f'<div style="background:{bg};color:{fg};padding:10px 16px;border-radius:12px;'
                     f'font-weight:600;font-family:sans-serif;margin:4px 0;max-width:860px">{text}</div>'))
    except Exception:
        print(text)


if sys.version_info < (3, 12):
    print("This notebook uses Python 3.12 syntax. Colab runs 3.12 or newer, so use Colab if a cell fails.")

class NotDone(Exception):
    """Raised when an answer is still the ... placeholder."""

_results: dict[str, str] = {}

def _ready(x):
    if x is ... or x is None:
        raise NotDone
    if isinstance(x, (tuple, list)):
        for v in x:
            _ready(v)
    return x

def _array(x):
    if torch is not None and isinstance(x, torch.Tensor):
        return x.detach().cpu().double().numpy()
    return np.asarray(x, dtype=float)

def close(answer, expected, tol: float = 1e-6, rel: float = 1e-6) -> bool:
    """True when answer matches expected to within tol (absolute) or rel (relative)."""
    _ready(answer)
    a, b = _array(answer), _array(expected)
    return a.shape == b.shape and bool(np.allclose(a, b, atol=tol, rtol=rel))

def same(answer, expected) -> bool:
    """Exact match, for counts, strings and choices."""
    _ready(answer)
    if isinstance(answer, str) and isinstance(expected, str):
        return answer.strip().lower() == expected.strip().lower()
    return answer == expected

@contextlib.contextmanager
def checking(label: str, praise: str = ""):
    try:
        yield
    except NotDone:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · replace the ... with your answer and run the cell again.")
    except NameError as e:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · run the earlier cells first ({_html.escape(str(e))}).")
    except AssertionError as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · {_html.escape(str(e)) or 'not quite yet. Open the hint above and try again.'}")
    except (TypeError, AttributeError) as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            _results[label] = "todo"
            _say("todo", f"⏳ <b>Exercise {label}</b> · your code still returns ... or None. Finish it and run the cell again.")
        else:
            _results[label] = "wrong"
            _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    except Exception as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    else:
        _results[label] = "pass"
        _say("pass", f"✅ <b>Exercise {label}</b> · correct." + (f" {_html.escape(praise)}" if praise else ""))

_CHECKS: dict = {}

def check(label: str):
    """Run the check for one exercise by hand, for example check("3.6")."""
    fn, praise = _CHECKS[label]
    with checking(label, praise):
        fn()

def _auto_check(result):
    import re
    m = re.match(r"\s*# Exercise (\d+\.\d+)\b", result.info.raw_cell or "")
    if not m or m[1] not in _CHECKS:
        return
    if result.error_in_exec is not None:
        _results[m[1]] = "wrong"
        _say("wrong", f"❌ <b>Exercise {m[1]}</b> · the cell raised an error (see above). Fix it and run the cell again.")
        return
    check(m[1])

_ip = get_ipython()
for _f in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_f, "__name__", "") == "_auto_check":
        _ip.events.unregister("post_run_cell", _f)
_ip.events.register("post_run_cell", _auto_check)

def progress():
    """One bar per part: how many of its exercises pass so far."""
    parts: dict[str, list[str]] = {}
    for label, state in _results.items():
        parts.setdefault(label.split(".")[0], []).append(state)
    if not parts:
        return _say("todo", "No exercises checked yet. Run an exercise cell to start.")
    rows = []
    for s, states in sorted(parts.items(), key=lambda kv: int(kv[0])):
        done, n = states.count("pass"), len(states)
        rows.append(f'<div style="margin:5px 0;font-family:sans-serif;color:#ece8ff"><span style="display:inline-block;width:64px">Part {s}</span>'
                    f'<span style="display:inline-block;width:320px;height:12px;background:#2a2357;border-radius:6px;vertical-align:middle">'
                    f'<span style="display:block;width:{320 * done // n}px;height:12px;border-radius:6px;'
                    f'background:linear-gradient(90deg,#ff3d9a,#ffb22e)"></span></span> &nbsp;{done} of {n} checked</div>')
    display(HTML('<div style="background:#141033;padding:14px 18px;border-radius:14px">' + "".join(rows) + "</div>"))

_say("pass", f"✅ Ready. Python {sys.version.split()[0]} · NumPy {np.__version__} · "
            f"PyTorch {torch.__version__ if torch is not None else 'missing'}")

# The answer checks. Each runs by itself after you run its exercise cell.
exec(zlib.decompress(base64.b64decode("""
eNrFfeuS48aS3n89BcyI8ZAjNkWA957ttVuj1jkTusbM+EgKRgcDJMFunCYBCgCbQ2nl2J/+7T9+ID/AvoOfxPllVhUKYKGntXsc
npDYJC51ycrKe2Yt3iy+e/v+fddbvFl8fe1deaN+1/M/W3x5/f6GfrXp14D+p/+C2aTf+Wzx3fXb73EjGHa9MT0a4NZ0hFvvP7x7
+yPdS/a9MMvCU3ve781G9HJvhiZ6U/6c8OeYP0d8d8Tfh/w54CvcZS/g7wF/9/m7H/AnX+lP+XPMn3y3P+DP4FaPZfGLjOZ4H2VR
m77EeZy05V7Xq41OxjVSPd92aG5d78Kntj7cYMbzNv326XJbrssX/nthf7l48pv75u1nn322jjbeYrVrn7rePovWncvPPPqnfimo
5gLXE71q/eTH+eksKg5Z4rXjpMB088Ou3T55V1fUrfefvba0hF8dauETD13wU9ys/nf2xsWz2r1wNNzRMw5Xq3ax73qbhP6nv0Wi
Zq7nUuy9z3HV+0J93yT4UFd1M9S4amdfff/7NIm8eOPJq3sMoO9F2zzCFdPk3jRk2kmeaidpaKcc0MZvUytZtRHCTrSRpIW399KM
v2TSSuC9omuv6Cc1hraysqlteIfGimpjNmIbVOh4/3zlFd6F50cX2C0KhzV6pXnBCEaNdb3VYhfn+RVQfrXYhFe+ar+6HIR6Ci2t
gVTQjaDxSjUmMH7F7elOi/QhSvJ2EX2sTQBXetv0GGXtTi/fb+OibY30kBTyUt71HtNVuFQvx8k6+kiD+v146cXehqAYd70jXfai
5LCLsrCI2vL8H/z8z7J3fouyNG+3t1EijRKi4rs8qeaDxrL0SBOsNicvXJq9gOe4RzM36yaPcSP3eazVWzykOffCd+fH21vvc9o7
NmR+NrDLQqJYP3dBB46LfZpf+T1arryI9vnVoE9ft9kVUUTV/8+0qqeCJlyk2eq+V9DY0gyvr4vTPrqSq5ttGhbjIQGg8tTJ/RS3
ezRNChh/7uX34T6a+7fOlwjvo18PcRbli7ssXF99yA6RNLSsNdTu/MkG0n05Pfoe73rX63DXnhM4l7cMjW2mxhzFd/flw7JVCDj/
DGZWmTl32mbwdp4FKl4D13MlGi2w/FmY3EVtXiwLQWjYPHueWrtjrrfViF95X/fydFPst4e8fXHChfbPhfdfaRU+94CrvV0UJrRl
luHq4RhmdhtoG/21Kxv02FtHRbi6p3cIp/cn+rtK8Yf2OM99aR6wCGq6BOYAsNVtS7MHjcKfz7Gzoo/79oVNhH7uWIMt2yP4xtjT
REBOqkV8rbK2fZW1ndSqZ+so05LFHS1B0b6gdx9oB1218iJcbqNWR9EueWx12IEBneb86q1wJ37iQbfDi+MLGSC6+bl+QFMnkPVd
+BGPWCztVDI6fuJBT/DvxMmwGGEJsOuu9yX1lkdFW1OKEC/aF5ZVWorBXBO3/BL8Tn78C34QReFvwiwI9rrb+zBf8GKGC0Usl/rL
6lAQbZiccQ0lnIXJqV0dNXgHvcQIvAQCq6Y6fCXEFdXLbcm/iwXTSHzQ0mHnrgitQs35aFMB4OBs9ETJX7BOuPdKXmgnxLOwAekC
vVxdiC4/eaFepF7kWfrc7Gkci59ubr55zyIakUTIr0Ns2XYwY2F22MePQV9+zOSHPMYy2MDnH6O+vDPlO4E8FvCPsf3OqPwRBOrH
UD12+9nizQ/ff/323Xc3X2E8EJIDkkohyZq/Q/U/fZ8SbgVjI/3l4W6/jdaQQcLttg0WTBz3LlrTWoYHog10gfgi/ZQnFYjBeFlE
5JsgFwCQehV4JE/bMJXmsI3Vt89VK+W63mVRZCMzvlb3aljdq0tBZEIriEqkIoRKqPwXr720NqCWGpjy0PtMzfjZZcdQJHN9Lu3x
Dl7qHyArIibcR6uHhb/w23qUNIms8FbbNI/aEHeAnXHa9Xh5W8V95OGql24In1liWceP8ZrAtzx51btJFK23ET3BnbRoaf968+ab
9/OXfs9/eYsJlt13vZejfs/7gSauVqMIH6L0kagWU/WcBubtDqt7/B31vSJePURFTvQ6SujhjciC93Fy13tZykCq9cA5uWWYRwvZ
dSRWApNbLDTxLD73vW24JFpBy7FWU9QzJBawpHHRHNUoKlMLalMjGfKl/+LSC/p9vEIbglC/511vdwBURDM88bi56V26jrZE3qLc
O8bbrbeMPJpVlDkmNXBP6pCfwIehdfKC+S/Q64B7bZ0/v4+yxTok0cXnpwcYJIA+6Ht0uTqzQW1mtB9f4gXCgzQp7rteuEyJ+Pm4
wm3u4o+0NkT0jnFxT+gQZtsTCdR9dIJZ5V5EPBPPOuY3rM2P9mDU3sYkfxfpgtemPW9xMy0aeB6tDllcnPBdLt5iL8xZZYQID33U
PEVwXaU7AjItM9aX3zAXL/zWJzs2/d0SuRhwV9KTdMTL6pnuBE+8GB1WQDqsgZTI2stzUIxcoGD2F/8WtVvvadSgGG/+ev39X4hu
7k7ENejRI7HuFkbWyuUJAGd1D7a9xtcdQ8s8yQNnbYKoW067goCSeKxWeGkCsuRlhwTYlO/DVZS3nhgSd8sNhqQH7PYlDIjd0k71
0GMVu0Y1UIycoBhrUDyCPbTuSYKTyZhpyMRuXcSMNvfiMVqRyNmmZwyMADBpyCNmjqeokcdySYPbkjYwwrLOc9iGtMR4/7X3Ur33
EksMnZRwHtsZWNP61EgU3fIekvSYeJCydfdiN+pz94dE7jPgvHC91vTuU+1/uPn5w3xAoj2kvmjdhuykrmmdkYUxrTHO1c1PvTGX
cfHC0IbPonCr1riyrOPaso6dyzppGyFrIzxyGSc8oPZ3P3z/4a/zX5j/3YKmJCTV3RX3V37QcW9TWxpO8wVIHBMozJIfoE46TeQe
eivWmN8Q2G7iO6JtUJn3W9IODsv9NiVA0eWckP2qPSEU6U2VGBp+JLUiaxvh2B90INJtSAJJt2l29e7mK/MgAXbxUShKS/rrvNbX
T+p6jYbkSkznkdynR9JTLGBPasCmkb38MiqOEW3kkjgTwQtGJdUGY4Jgut32vBszdW+ZEXIRg02IZoe0P5L15rBlpkfw0uy35FmE
+MSEHXR86uRTRVqEWzGniBGUoDTkz9ZYJABC8gKcHlocHiCpFpJ7SNtux/f81pPNsgm0w3KpYm2m3QSbp9Le003BDNQvzT3K2gMJ
d8BNH2DEIkjss5joItMV5nuVjTCtrc2UXvxAL91H2723jvKHlwRs2uA5ia4RrQRzEAVbUBOhLNEyTR8A7L8f1kTGQZjjoteqQT04
k+eEPh9IiiXqkGwOeZwm7VKV0VxSc8rKHYuHdrTVVv7jXQTWoxmc2jyraLsVkCqlk7bDOt2RkrwJD1sSKZO79kAQ+QRrGnSZrLe6
Twl+beHZt4yY2DvO65+cm7QrhG1n/WIJVtRhgmLdcmqtWFATUgMWUl+eQTpohnQRJap/A0/YwjvzS1B1C8rmqn9bgpQ2JPEoGqcA
t6uZSpYesQ1j2n5MIxifRfLeHnaJvgdrcbwi0T2pTCqoTQriaeAV2SGCRhgX8WNEwqOvdkcS3YW4RECv7BdSB+Qd/UAusvv5oH1D
XM5B5xZhiZcesnB1ar+CD6XDDgYona2gO5uwDD1oEGbNmxaGsk+C38abQ2wYwGodh3dpEm4rsBnUYDNwLviwecFZpVjtsElqfh+M
QDN6kEqtluapt6TL3jbaFLxtiDxlkeeaHTdOU2wGiDWVYW0qECzrY2LEmc1edL3oY7gqQOdzjWTQd9KeBwIl4psMUDAsFsJ/vE+3
kZeTkHFyLO7Iubi7ME5kEu1gSEoze75IjYY9gKc0FIIfdKcwN4gucrbcDPJdmjE8DqLDtcLtkTQWkeVbwt53ewZmbzaaig4CyFXg
NKrBCVLnbNSbvvDCOxoqaWgEIOC2p5HL4nnFPbGhVVjQy7kWx6ifhLbH8T4mOkiAOt6fylc14LKUHlRqJAQ+B/jG7r0Be268BoLT
xiX8hJ1mPmchlb7gwi2LZSASPvrT8sOlsYCwWC9Kdqu5h1fwU3bYYsd8rm/U/6HGU5JBD3uvSEWJNcLfc0QmQsdBT5t711l4XOzC
Ios/tsOPXQ89K+YIdFHApo3LiOAVEa3Kp6WhoCZ6Bix6vkuPssOOWLtSlPGOYd5V+F17QOnkYbx2LNOktkxwkOZFmBC/yo/hfo89
DgGaOEyW/hYl/BVCKK0DSEZHdiEjNORs9RLB+vfWJoEys9m3/hDNjBi/HlKYP7C8BdcR3qeH7JlPajOHHAhwFseUyMXmQlM/i+Ko
jnveuygUdsKaxiV/JSRKeZSn3AUXxila84wW5ngfsTIdM0i9DBZ5B9ymT8gnu8VjqPjmL4u/XWu/2Y/8HdScqSmuAR4wrC6JTuCO
Qm7R4Zz8gZruis/01VPta1w3gLIJkQXoaQ3QEOreljQhtIw8dWsASTZhDHF0zXgBMl7lsFhXMHKbehC7p43NJAnrCaiRuFkUUZaA
eBvKTbJkuXXOxMNBg7kPPuCYpSfYV0UoJ4L8hbIHt6pSgsDEMmAppiaA3/ex+U2DLIzbPdJ9mg+cwtSw2IdSboFnQQgZlySVSAys
gaSG5PfA1XTDjNNyLFuLMqjJbQOn3DZoMAoqyzEHZ7DsMFVciY1orcYX+koX6PcMnRR+dCG2JdksJa/AkFk+BpjKNqowymwYEXjq
uqAHCy+bU6SFChSCGhQCJxTcIth+AeypLr7AgH5U17kKDPVeDWxnA3e8ykvPujoDScDfsroTdKMmYWmozHRQmynEtmvRdse9yQve
SNP+C9lRY4VkLKMqEzM2U1ho7lxk4Vr0uSIKd97ycBJSy+MCOupp9c7BOXSDMxC6wgNQkAqMYOqPFJRYEDxWra6DmiQ3gCTXGvVl
VvS35xG9UYotJCIzKUwmTBS7MFwUbxFdJh4KL5HH5nmt09NsiW4U8Y7Uz5+0/LKMQF1IQd1HyTrX4jPU4txBV+pCH8usewuLiS3o
9VS7m7eyIRQE5eRlAbs3pLdD4UQzsdybzdbXm9OC2qgGNch139v90HrTjmF4hGr7oBn2QQg4iFeKiJaLoLdjOtsFZxMjpiFOED0c
yNAgwm23GLsvY280FDdvMjSwZ+WRHTy/kPKoXKqakp7T5K41c4XmmpQ6NqcFx3ENjixDCbT8vsLCsmn4OdA2FojdlRAn+OFwC502
i+BuV5PD8MqROuBnZCsOWbigCVMb+WIbP0TMuDuv6c5cXNrwM4OBdzBcCQVZqcAb4fH7ouOy8ewBSgl+Ws2hk6/mgejgvkFUgctl
FYR9CB5uE9Qe67uwm1RqPZAI8oCvNrxI5BV4T2rwhuT2YwlfDXKFsD7UNSJEXvlIA6CVJNLkIxs0mOkgMS0hy5faGqtr8nMMn3/Q
NzfcUhO7zKKjxwY1lwsqK7tgbiP8wwgcxsTgQUIlokVzTqKY2Sr7JisAnNYACInsWktQq228W+bYsmINnc16oxfQ1baRTRpKAK9T
UKMdzYNgHG8ByRQ2rvykfEaaExOgQR0saU2sjLILEXNwRi2HbiMdW6hZIEV8ZVfE2n4P+h3LpsYtoXxatgjMdOs+i4ggbdd1OwO9
LZJx3nqiUxXb2e9NRsZeLvLxonKLTaqlG1ZoZk463J6gjFXCY9bKDGti2dAplg2fMqetdhMOFxDNF0AYqmGYaQaKme1oAddOXJuM
0AIsDhwzyw2USx+INUUvP/+sziKozULsZ3hLCRagquwMYTs7N3AuuOEh2f2XChXHSlAZEks/h4oR0zRR0/G5WnkpF2cw6nRcE0dg
cPuM0jkIFSAyK0E6FFBaIBjUQAB5i0WimXYBD4dK2iKxS2mQ0FSMWsT2e0Zs5leBSC62+uOAgZGt9qQcZQiKoaGH6xPpzFG014hr
4DIHAlrbxwWVLOfnhOIwTxbjiVp9TJ7FLIUG51DF+wqZQLwmsIsSDROP4TlaMZT4IYjSQas6ncZ5zGa3lcFzIHZCvH+fwwtni5Uy
IZmGpj68H3O2fdWYGLVhcB08rbLMw9oyD537dVT13M0ZnRp066LTmV8SbFicZm8BzYSIKvuTof5gT/cRze1Ss9guxb40a41C5ZK1
qF7ifVj85d3br86pnLyzSCB1LDIo312PIyNLQayI9hx4WMKEw4wVC6rAZ1SDD0TMdyEMQBLLgUAwkmbFg3ZI2F6h9r9SX7kP88iG
RdCMxhCePBJ9HVvASJRZprwmaguUwYFV9FFI/67rIa/AiiHU9ELTEdfmoD7e2ci1J1HpRxv2Fdkv3EBPMAIHs6J074mlstfrwW/z
KZ+Qipb9teudfmWXkDzTHrkdQaO+k9LpOXIzHY6xXTiuckBICiWFwJ2ly3BJDL44db37+O4eNkY2aD3TngnyQdJy6QTGA23AvUuK
Q/YQZVettKWdwD9ef3vz4cPNvLU6hUnrtox65U3RrlIuK/IVoZdWKKVKGEHAeqeDkHXzIJzIK5ZI2vN38wc48+c/yt/8ilNOagMJ
oei06PZv7Bu7GnbstsKEaAmCFInBtwvwYDRKk6NP+vXxhJjGK3hyobXjxyqF9ZDmvNnkpNAI7FsIv6O/ANnU6QVXdoxzN7jBM+te
ERckFbCdmAWPl3mJjhd6dx2yx+gJK/GwpuEMWcP5wNuX3vR+i+9+I65+KVQmM9oACUabIrfRn64UNB7he/z0XRaRmEjS+P4AKyvp
jev0mDi2tVF01G7mrhd34d6B3uaeuA+XedvCbSGzrBsRghNG1O55r16BDekHOj2oTRycvb3ifAfbbaKVYGuOcFa6jDfWoPq9fq29
isP1kKgAJCaBhYEzlNI0rdDXSW1hoAr1xe4p75Q2ie1JS6DSUTVKgbtC+JJ0Ldr8A/Ha3HrljjUMI6oAaamV1ZbmGW9ikuyl55LP
/P2Qi2aQs851Bz/8+dIateqcBFuL5TbFIjQYwbQ/zt9hj5OIcisLpqG6BJWylmeHYRDbyUWsKq0aNF+SAXLIXNN/AEGr0Y59nDxo
IgZi8fGReDrxczTRRGa2RBsuLoggbI9X/j+MFPxwyOreosdwG6+hJSbP8BkNa+rjEOrjSzi2lbMO+xMelBJze94b3i/M82i3cPe5
dtQwLpXCxCaMs9feqnwBmoo8GsPGxpywxDEY8U8cT7Y8dxGOnorZDYszOZLVNoTD+KIzWcEwvi16t/5MkyMTFRNICBm0xVI1MUoJ
wrY7zwyVGdU0xZFTUxw1GPCZE/BAIArTbElMIJI3YXav5vy//5cJLKoHFUn4ysHlPcJuU9mNegMiWjKHlTOrjD+ojZ9DgGUkDO9J
z/sJG1QNh2N7oo9EzHIOtUDLUXiHOEwSIeFPhogS0rc1bLkZyVcObBh8ChwYxhzKxzk0gnNgNACgNFuzYTler7ci6cYc5h3uI7Yy
HSvwGNTgAYWRtSAGxiWeZ+sEpipktjRWwwNGUuADG7EUV8Kuei1k38Il0OK1lrwdABo2bheoxs24O2BDyCdwpzLfYW2+0JxGo0ut
57JdXhRgcd6GmXUDE5jI7cAxi1FzLDtmoOLdLKmdIy7klgqwM5Fwr71xhQzUguyaKAFbjnkfsJeHWhVDn98dVdyjo5qKNGIViZ8y
0RX0shgI2BOhozm0uwxo6wUIxpDg/573HbGxQ6bW+RgKApgQOjjEcDv3aIg7NtXH4ugQN4YDnm4j/cZXniykoZ7bJdF6msQrdsCx
Lv/FmEFo+Guttba4P2TrwG1hLRDdZaFJvBnKp5HYchcnN2g2IoKKVqWU0k1tPMe0I34s1jGeMiIVnU6zNuLSzglcTGiMdv61Lxis
3lSwCSqwUYqMMt7rJqhrODOkDeKCBxK19lH4oDZMBb3GNfQaG8cfcFOmUHF/GeeOGCqpBzRdWqO02i6BPYx9YB5iIDbJLdkBNO+e
Jt6wR8+jmVUwpRAZMYxok0jFGJIhAaldsYqo3PzZrOvNgD1B57bj3pg5BzeoFTCjbbCQ/Pj2zTcu7wWtQph1nz2WubWKcSKR1be6
f4kIKTtmqdTimxjf88TRsR0vpIXR546xEn9dF0XPxFYNgdrjNLvCElydIquZqUNqteRQnnazEDqqaT4jaD6s5fJ6iu4DD6RIBF1A
E2LuNjpp7UhhcM7hIwXz0M02TTPt++P40lrsseKiIVFRBHK3zrB62hAqsCSh+AEho194IlvuFZOkKfje//kf/9Pbd3DFkVUDqgMt
Z7BfFaUpbvAC4yw9GtzBRfQYJWJHqNCAaQ1WENjZLQZ5S6zRwQslb93Df1hYxp444jgb43eVPkN6g3XFFeulxzSjd0m/T9OHMlIA
cX1ViiDmfxbctucBkeMmcR0a5kIylwljOa7TH3SnY81Uh66oE2bB+YJtbgg2FeIsgop6W3NYagDQr8Q9jmsi9phFbGwZHUsMVYaH
xtEqR29FFOyEeAOtW9MwJH2r511DiOAAHfCqJDpzZ4vXyc7gUy+w42g2U9lgDpg1yPg7mHE1zDgjVZK+BGQmN8uecFCbMGTy4Uxl
PNQHXEaRVUY9nFW9fo4BGyk8X5yVEZgHYPMXvjB7WrGLQa9/+0TVgJOjDX7bLxvC51NtCAtqt6utDPDqwDTV3EY9mX6BbHqam0mi
d/AiWZhovQCOtOnhLmbC/XUMf1K5+iq6sOIde8nIha3J68CYxXychS2sMOSgYev5PUuwhXsieh46eEINzFfCIysYW0gQ2/SOsJaE
PjRewa1BDbdccePjUv+QJZE0Xb0wUl3hl8VXN9/9IBUWZjO1Mp36IqinXnnv+ctTS7E8xNuC5H8MuRQNiFnDCam7Q+I+Ee9ADPqp
pkZMMs4DuIlHVps7pQeVpKshL4ye60HU6c6wBqqhYm1sKBEjizCxbbzMwuwExlSOiPVkCVVn2gsCpRdsk6U7K3S9dQZ/ozlhCqXr
RFUIWXx4N78M+n1shF/sH36/LBgy6nfsXbX4k++6ocnyOTXIf5XldFxuESRYbzZRdgnHUMbyHLWJoh0C5i1c7US+z5ehYfXYIYL+
/Fp/zGvjUHdYxfFRbeFGThw32tQR2e5hFUYCHMFpt4fv6Cul5cyBqMpp8BVpW2V2iEdRD/+QaAiIwE/oUzoKWX1w9wvW95Tn8ryn
P9GP6oG2tcyRNly/N6i0ozUdeBVMS0RoisoajGtrUOo7IYxI2dowfZVrrcy/MDOKCJQehORq3mk4+ZLkiS3bGkkKFBEAgkOdNDs4
3sSd50EUabFnhQQQtIhsSck3fKtJuuF3dWL+oCHvx3RDwB2AoJs33YtUEf04nRnOvgqQJzUga+FbNxESAyIZz3YZKnEVgQ8cPilc
g4NAfMielVGIOYuGtt0acSvOaiPjAE543KQlDhcBsXOQNWckPQqfkIDIZSsW8fojDL8DSIWf87K+qgmalVAmSbjUUiOJQfs4yp0b
R5fbMAahkRhHthsFAPVAJezFKZ1Na0Cfcj2IFwgfJA294Bgql6BG4xOAi2tHJaTGH0vju7YRqX22TiMXEs8arGpbKAFrNq2w9Qeh
b/gjcW9flLldO/jxV7S6l/wYSyn4wsBkXbg0rHFkpAOgSVgcsnCrenPGXleANqsBbWbIwWz0glP2PT0DSZrR1EGlAiPiSMgBVMCe
9x7wYhztKscFjTO3DK0ArHLfncFw0qDfwK69AkfyuXyMDnlm7MLudkXVLBJlLaPlffVKVZZp8c/dgeSZ/TamKflcBmQX2XxqUlNs
JlBsWtQrg0N2ruw6pRoGrNAgAyn32EaBrlVwsLIWpaiFUAZrvOR8jsQOUT7blZMnswuIPm0Pkdjj+0YZ0MFHgBIsfoGuMYNxonCI
i1CSZLFASp/f73MSHQOXvouL0ZTyOI/sndQUogkrRP2uqHiZRJpKcJYNr0FvoNsukzzlOUc8/KSpSEmcpLvFfrdpD41ziC3t7KFq
jTkIXm1s/wt/7MSRoL8IiEwgDmIXFvc9WowlJ30/QF7G3iPUeeCvAb7SPe8CN2Fveyiri3H9zZmEvSGlDTEWtLeL1AumFYANagAr
0wxm4xeS/qoMDLzZclF8t2txGobJiWV8kVPLmD/qZzZ4ofFLmyxzsxUdQB2emRcdoTSTTo/BHIdbSYXnALd+v++SvfJ4p2omsWRo
UCmPaJ9NuoxCGRtL/H6Zg5e7MzmEB8N4rkPUYRaEBzvocx1L/Pon+jUVoOuoQAZMCfnnmAVHwJ1BaRa8V9UdeAIkJBK1u7JKlSFw
NJBsU0TL0Jdx/8mQnOd5s0tjnynmoL0U2ondaOyb1DSiCTuL3gCQnPZEuEDYVSo3hCBEAkMi1jH+FjpeOSSueL+LoJ5y4SkH2oya
6PNiilpcU9snwYSFL/uBkOxpmfj1mskN05bqDhnVpgIdYVpGYaApTT3EcvQhTQmmYSaTNYEnx5Q40ipKCiUxOSZjFA1cpRVvJ1aY
1MfGXTG1dkVS2RSEDIlpoCz19ushTIp4G7U/Mj8iAkWYc3a9H4xcdpB4XdwzecYAB/2ODWF909wtw+eVV5WZuEhSuSy2QagqJR/X
wF6qBQPDCdh3HKorOoEPozkkiH5BWRDcRHwNTC5GiC0f5rgS1hcOqwd2SC+1NpETNBD1nKbMqXwXJ5g447oFNRaoFBat21LTmCX+
sUo41mbQM1HB9b6qa4yKK4PBUNKgUdBNspRDlbBu5ynbjaGRBYag+0asAmvVm0NSL6QwqekJEw5PGoM4XlZsFJbdn4V9TThlaeE1
51pLlXVRZTQcUDTC/obDQ8oQ6AKZEBsSixZ8py2adi0UvFIixxTY4RfmeOG8xo7kGwRTVLkyXn/oBuVg16TmiEi0PbX+bG8Xzu4m
k6DSYZkBYncmAWPODnXZH3RWm0BZ8+cM3gzUyhpPa2sMtSSY8vMcLowXYP8WSsM/Fc8nBhDTEO9ICAg4hHgwLjN6FbM/8pPHdLkk
rGQZyhFCNm0o5ZLuF5yg3baLhfvKqHsRaKsul/8KkUe+5Opf+FjDjxXwnaX6zQnmMAsig1vZ08roV+Vl/tsPb66/XHx7c/3NPL6V
wsHKbWlqiv7Et+uFReeXo9tb90pto/BhQZMZdSqey5FXHQ1vF2m8suS/Vwq/0ZCQ9lQWBJNvSGzkv2nrD5I/kGZvd9sSXXW/hbFR
ymspawyYNREPCyWmNf1iyvrFT/yS4dGkYyQFW4SoTTjh2ByjI6KBL6tDXqQ71DakoXNJEKQ0cbLUaS+Ud3emVEwblAqey55JyBcs
35RMhm+VyYbKihLo5I7XnrriSEWb1hSEKSsISt4dSq4KlISe97UJr1CRLVK/Uit+uhbGUeWnr0j3cWD5oHluErxS9Z0DE0rLnA7H
oqdPxlBT97i6fG8wH7mar7fMTz6rZTNi7588u30zQI1cO+3WsFrLVwgD4koPAtHKggxqCzJQlilr2vyuRLxINFYGxXePL4R3pbLv
GZxniwgrtR9pP3onFCYQqxSU4PUjiTmwdq3jXLLvcgdeDmvxy2GxKKsPnXOgyu0ekch2S28I5B9HHwuUb2FvcFkcw/GSKrqH/WZe
AKFDGjXkESmYJ6UqdlFSKfdBJBOkv/WcwVX2M3r6GracCuXwpHKkj8iq5IDsaohr2bOapxfzELarI3u7gG1In2UUXde6C5UzlcOa
Yk6x5PpsCqMr6DKsoYt2tQDAxs+rCxfmgKLCgVCBJ9yKK0ZmqN+QbmUUmLXM1K5kFD44cGTUmOt4yMElmB0ZusjNZ0zUzTVGDOZU
JOdA+pFwEhJVPUd1IF58BK1+/fbm268WH95+d/NeVTmdy4qq9bS7eLpbybRcRhts0nKpK0Af1YA+4vhhAp/OTEQBAhk+aYiP8aMu
9VYminJ0AfGFItJJNbJrJRjMQTrdIWwxzJZdr+0bN6JC2LdfieUDCe2seDNl4kpiSnCtzGhcmxFrF7D+bIi8lxnJ/IVZYJqUnrzr
xCPFI2HRNMP44juU65VpkXBeqHpY5RCIExKtuTvEuYtLNDgb8rsFl8Kg6X6/+O79X+Yf3t1aEq3yPnS985sX5m7VOaE2pd5tuq4d
bPAtR3a5OUpCevjb9S3sHWN9fETHPWZen+emZDKtQZwJd8HNg7aX3KOyapPaqkE7ea+N56AvOYwZWyyhCT/VhMikZthbPY9IN4c9
uPS8gmRFMJNeG4LhVUmUKsnlO1ZyWjViqVKjvx/lDAlA/N3Nj9/+cn5URNb5g1Rw9SAvDWqTuo6U+KNB+eF9vwDhtMIRWITTFfYi
WJhz3uBaJFS3NFnIXzJhcDAQBMQxR17ECUTMjtgoq8Izpxhu/pwM7fdva1VLOb+urwYoDDtT5PoiTWr7eFrDiCnv410Eg35M+ogI
J3z3kmOiSH54PGltgL0JXCzXhoISWkGtoEYhJG0TwYuhzZy8z0uGJ8VF6sgwa3Ad6NL6qNWreT2kyLLor3XjLi30Daj/CEaTPAc2
R64VkJQtfQQaQ7Rn5ZIKTbehx+rRkmYVolG/rO1j5wipUsbN7aAR+/1jWimJnKv9EpbpR9bCzWoax8yZDTFz580zupsDDubNgCRk
hKFkjyyhCOrh3A1bQrl1yPzxqIs00XOcQdWyChRAFFNySy0YVOIWGUDCf7p2gTGpq5JU5x/U5h8452/0h32j8W/U6RHb2h0KJpdE
qq+//bZjl+bfz/0JDDg0kz3ttyGCOBxOHGqZAar9nuURElwlOSTuQh/amMLXlri2VNfskCNThYUNQ6VJgmWbkBXHi/WByNEKzmAQ
IYRTcNlnC0SDGoigGVyLA7rsgB3JbKfQRigJrtE9RuycjRK7l66OJOCtzzSWvkAsxAKGvM+jMrGfadAxLdUJIQDbEGl/Z0LhrO7J
qAHyww3z6ZsKJKucXYPyfJWIJ0aVVVKE3sCD2n0GuD+8q+zsWquk451hg1RAwetSOl1HdoC6biKurq486tYCDmsLCFkdejWXY7ct
BqKGEI3NlQrDQ1fMiXoSQTyUlBytTWyjZE2NmZRHwoD0jsv0OdZk1OC53DOHZiu4bWIQlxfSTuCulrJfEzEqvPaktJU/rcx1VJvr
CPUm+nCB6S7EsjwdvFAONRXWzGBD2Uk4pbRh20I5VR7mEzg3fnJ+i6VkeZzNUG6Ig1IXAKMJTtk1O6xMcFybICTmaWWCk5Ge2qVn
XCLMMWWSDhY5+Q/Rt0qyxgEMWqOr9tF9df3L3NC/W0JsuaCooE48tfODJasApIRYghXbH0qBTkVBcLP19CAWa30IR+d8+6CxMsWX
dc36yRSzmrQ7Y2n3eyFcYqi1CV9ZYyvdrrXRCicklGPWFRqVHKyqveKuLrlNy8UzUJHl1UF3JYogdq1jGa5jjoNB7SB6AUdatd/8
t/cffvju5p33ghADR+RJMe7GR/7Tledy6N6RbrX/hzEpBhvB62xwlYWt9+kkjvwQQNxEFsskOIm6VSvHcK2s+bS25pBn/6JbX55K
GTWLJEtb0irzkyGgHMahKL/dDZfU0/mM2mpfacZmeucl7/tuk/3+uGBKOZ4wDUEskYTBIJSIuhtPkCBVVozv10/J6bPod61CB7pW
zqAJuXqyhA414LYgryIubbUoy3X6qlQ020vgKB9IZImyFnMx6OYKdTRT1aQKY2qjjuMFZq7Jp6Q2DJWKNZ5UJx7UJw6Z7+sycMKX
wAkVk+K/6OJ4L5Txg0iuNRCSVCCxKkvF9lSt8sjl0NgBt0FUEwFxeTi5QDZoiuQp5wiWaPG96lwG9bkgbmQGw/k1dx2SSrSGzY12
RFaUOgEs6Wu+ITXjtCvXNUZ3SfIsfIy2gnOBhPsFdZwLps6ERfXqY77YH/EyIiUEa2eydPUFG9YnOTSYGsy6ktxZxVRz+Aovmtgc
QFYVULl0NdI8XGcf9UshJV5/PCeibJSxDk7l0hdcQUE7RPNdMxslnNeVZqj1Lh/oC506XEVXYoq1RFYotu3f4n37FTV2SOJfD1H7
ww8/vn0DG9CcuuFYa0QTqPNV5ChGNQxlL9+E6ua5weJ3UsZhUXhUcUtd7xFynXmjR5r3Lm93/uh6DY/SOM1DZ+aDeGUsG4QXOlyz
IbGZ1nlUX2fYN3+sLqWkEdDOYosmASSHOJKD1KtK0eXpIYz2YaaKTSshQhcwtWocc5H5s/L0DsQY11wRNdD+/0OXhKbBFcY5qYgb
ez66yKl/KQ8C7cyrubNqhp1zH+syvlvwMiPSf6/FK75iS2ziarXKZuPnlRqsiH5COV7xkYoIKHZF9d0JsaEPy06y3+paoxJVAs93
nVW5TKpn4ffuHg0Rhr2Uz+PD38/ZlEosB2NpKl3No1A1PmVXCDdiQYGQdsdhujb6j+voP1blQSqEHLIfyzmqvB2RvcMerrglR6OQ
/Mg11OPCsB3ve6TYKbk6LsqlSeCr4ba030F7jLmeDcmlrk0wabQF8b5ZIACd/SBVIsxuj5B+raMVYt7YMhSr+hQwJvwX+3Q1K2JA
TVbXzVJL4qi/Rv0uCFRR3v69FbYuOc17SX9Hf8gJO0G/jEoIW6oWH4ldmPBZiUCVCR9Kv1nZrb1ik/qK8ZFI8V05bjFMKRVTMR9s
wBwYTBOXh2KQHtI9gBfUB36k9Qd0ougxih5arnUxsv8OY9JUxyI6/8SYfh6ZtJDjGuQc1Z0cyiCVKw1GS2V4k3qiyB/eEwLwfN5i
vfNp7tLI+nYNjI95kOI8yiqlDnw0c6mu4LS+ghD0WekiiYE0B1XBoithtqyUsvJFTPLAzgm1QLxWLFIoIyQt5Mkz5IPFDyk67zHx
5sLOknGF7X2+nr5bzE8W8JUjBrOrpOaygjm0WFJNrrlQqfrxpavEM15fcDjuYDYhqjZUZ1jNWMQcVsps+GfHacp5mgNPDwTO71ya
krcvhQDiHAHHtAL3tCSuo+uxIDji8r9I7h42jt8qIfqFvCX1L6tjD+pjD3Q50Ym2BI0mL0zlhjIXplt6DcWqlUdbJggZM5yHk2tu
DUcO2eOFpQfGrJnME3VNn1oiPD6cjZk/9mWSFbhXZzuoz7aMIh+NX+gqZtXK0yJR6RXjL4waElfDh3JUyztboZlW6pADGkZvuLYK
bLeHfXM6N+34zms+g7nprrw+n0v1TnUozS1X4aYXq9dJWBzflgW6z+EJgLVx6DOfCqwxXyOQFir0CRfDviuzwT87VbPhWE2/wdQp
KyvnhPAZvu9urr+6ebegcalvX0qygBmx44HqgEb1AXFYgKyjZ9ph6gMLpnGql7caEENVt9FmYnG3KsNZeVItrCFNKGAE5n1KQzQw
f016M4Y8UsQHhyKP9ffPsTj2ncFsqH6dg/Mh3O/DBa8ZLSr1ckFNcw1xX75y9v6S3kD5pnLGOllJFWIwN1oNXbSbzqx74jQ745OT
IAX2ggo9gWzh+dVVHNdX0a43Mwn40O4yChBbNQuP5cB73jcY57k9qrLKTxcY8P2GEAjYI3TWwZkFm40VaZapeDw2yQSTqiVbvgU1
4jypT3mibNllX8pYT7Mtg1XAL72f+JQrBc+dPlOjDFODvJQrAxazXtdsjci0Ptfb9LaDEdRsPLfcSxSZM7V0RfO15PuzD1OXu4D8
gArukv1cEaP0639WlKq99x8Qp9ZPilOlMq9EKt2xmz5O66s65VMdxA7j0Nw5CM+2TknJrnvoChJLEso6s/B8d0CugbLv8j5Slrjo
I2uDDQeFBg1RABsU47GyLat4bc6awRNAYByJ99orD+wp510/0NEPLHvq0BwTw5pALigBoVqdP6RR93X1gAv1SK6foalu167JuS2v
IR+vzlWLfDllTSellBPmkxG6nhyT2pUT8zpylIoN5GW4ejg77aXw/Betp7rlitt9U94LqRftEXPqWdebypkift8+GpIteBIu/Nqb
9cvg/C5k6ghHxxrLsA38oA58lx/fbzoUcrVTRYIWFsgGTSDjEvaVTJfCG7xoNZRZlYYnMI626fNzasOHqaPWIN1iGWwSOLN68+p0
B/XpctHDwJzCpwzZ/rR26pdVzlWs21PCyO9qB/ukCM1ad59KE/SDpsOIMGUsYhWWqJrlBqayq/gCmenoHDK+goz/LMgM65ApbcU4
18w/Py4Bw6kDLvBrgJOTb6zMm6eBMzpLorRLiGE36s2h00kJBs946nORa556slOFYMMScZ0vE7bwb/8Kq72u6AtgV64w2FW1rX/7
Vy639fX1u39fEqWurWaN3xfzjN/vNxRTk+pozqJoJalsv+g8Xc63+WTr+kGdPp/U2SqPvVEnu+RFxMHKxoBmnUR0rcVhk53J5NE6
cQoRFRzjRLxA5DCYcs5iCPygMYhgtUiioxZ2VaHYQHLiqptgXJ8OZMlAcR+T9MIxOTi9QfmdUTWDfdblCfQu3C5jBbZScjpdtvXx
7t++/dvNAu7erschiTTCnxZy4uaX/FeZq5EGLc9CBoGdqkVTy1fhrqGGCg2sjCbYb+fS0y/GwvXfkwcOipX6eh3rHAFkKJSROPeS
+goLtRwB6D3dL92t9PtUN+ooIrTmPIpOtQfPtZ5RSx9fBDup5bhe6gNc9BlqxPqqSzypL7E+hVOWEOGdylhRxFxagFtTFRkO+YGU
ETnZVkfeYxR4y8u3MQfyZCm90uOj1UTw46g6Keh6RJuZqvxo19iKwp0LZ6b1hFoOb7XPHuBsiitOnxLMee0d4cmYW6G4/FIZi1v0
WDNtd3qsk7aVh8JKsYUp88irTI9LF+btYw41EQ6HY+6gkocE0WoSWdCeI1rTW3lrCV5UGW+3Z8c/i58JUD8k3F2rseHHULLhJWCC
hPBK/q56aBsrIl1uLGOnMKFmqqsykljVMimqCDOtIwzE8hvF0frVYG9dkwHnnmqn2JZlbaJYlyYkOLIohQrWvY+2a6mkTPAHQh8k
JQ3BDh7i80OuGWShvJhWEYasnRAqxUvpdqHaS+do1XTQKAapQ0LmSCZ/5SFnX9Yep7FzIPdPNzffvL/VZ72aIyCNUKAKRFSNofXj
P305//Nr0hvFuDLsTV+oynITnP4FRKPheDPX8JujalnsWODNHD6LmYlzBWC4wbKAJZwp1TEG9TGSMNziPBE1Fu8D8loK0UpVkBvd
eZnXTUDn3KnpUE8GutSWUTBfEcwlCB3J8JAzVloJFeCTiPjmh++/fvvuu5uvpKzrKk02cbbTa2FMvA1rMajPc6AisxDCoVZhMLYW
QZ0WBQJ9DB8jOSN3XWPljoVyy7q6QJE6YFXkW38Ig6ScICYm28WznpOSbDTIoT6zi7TFpoJiEJfOmx1ys8TqgzHbRCvN+n2uEC7k
e8OuQpQxrYN0WAdpKT6PcZb7mmloGSujVdoKE4PbKioKOYNA2IgDqG77qMTsiStA0dZhXx+QHNSHO6oPF6ZPbDzrXLC/H3Z7OPeb
96GRt/JKPeVPkY7zwf82w3nc8yl8w/n8cmqyf6gN+Z0X63a5KDNjh+St7vnYk66Qmd+w4tTy7Lkt+/2Gpm3ojevQg5j4kwyMq/uW
h0fj1PG1eG8f41AMN7T+KD8mvUlqKOycXOqNMQFsgeUiQghQKRfsnd5k2K5+PcQkx2PwHRUppoquUX9iWtNVbUI+ZVwnECoCJuwR
6afg/9V5T+rznuh5+/oIGOZXGLOaWJJmO3XqnilYZQ5VKAMRcP47W8iydO+arbvKMXO6BaEY0NcPpCDVyJSm6A+rw5/Wh89VibvB
yDgshK4vT949nwBAuDtSW79aYrg8O6Q6H0xcJEMuRExCXa7nyKk6OXW9PmwdC2odO6kPD4v3EdpbcB5HgwW19hCErLV9YCYXIG9x
UIAIBbb7AtfFK0EzRpwsi2dEz9Eiv6WC9nIr+bTVNcLiJ/61TOCuiua0C4Ny5r8pQQ5XCFK2ceJbluHgTJJ3Tnmct3ScK2f1JKtI
5oIgcp4L6v0XqQp80DPvKhGoq+N+zZy6OgwRcopJpjDj6HrV3rscmtxVUes2MtXPzfTl4EwrZ1RS7kKuSCahrRzPsuOggE2qLdxq
blaJ57JK1Y4rMa+c2GJEoId9lfYi5vL3VplDlBcpyeqmxEKhZf6OZLVe+NWMPaR33HYajvp8OC5WOz5RZscHHHep946WBCVRWp0A
y+ErKjW9Hkak3nM4jI4qKx/plagM0lcRQ3/qoA+/fhqoL8eBvi1MQoyhRJWQN30ITrp8jFOTYchYpk+E4eqJBQRzHXYuFl2ELhQw
TIcSvYq4rtS1bHUZEOcSgkWsiIFHyXoBczEYFjCCpAVwEXzFXqc2sQgdK/seC2peIyD9fr6dXTuYvRMQiJfpR159VnxYhyWOhdjv
KjgHdXAOlMqseqNWDBfR3dFFUoEriTsCS+QKlI/ZJ5qJcp2nhwwV73E+lsln5goMLngOm8+l5TTZOr7q8DV817YIg8Bm+1WrfmJN
+VEXIrsadHhLZCz/EOQe1ldjqFZDlbljWF1a5zvwmRhJVVHtluY2UBhtheFwhUyKfDU6O63TRXWxZmTiMKvp/D+mSdgdq02dlHSf
syBiUf69pbKfJWDukg/moSE9RCfsd7qwerAX6GGOI55bvM1xc2Pf3NDNP85Xe17CY/5wa1U2jDa3ytOJ8F+pN2mluGPJFAYK3z7D
7GOcJKjqoA71oMlHpyu0RWpDWRVFH/ikz9tDs1UcGtVxaKSrUFTI2pITMrh+hGGhGlXsQyqg2+RIaDtXe4cNRlkcYb4PYzkmYcYH
as9GcowPOwtNYeHXpJFNJ0N3oWHr+CPVmCwnjlSqFM4jpXUNaXgqtTj5sCMJKx0G/SpsxnXYjPkQBlVPoXJM0mzUm6rzoYM+uAL1
w4eRb+hB6kvBilBaTr0zgUNb1tNYHYSYzMFBKjKI87pVoQbH1ps0aNQ0v8UpglN2EvTVJDECHCMmxSG/0PCzJzupT3bCh4AFmKQn
7dln5ll2+68yFMWWYlnWCY46/okFZc6SYAFNabdOF/PQrjpgnZ4riXltcxpSeWxdp0lAoVfmrV1YZPFHFglRydSXc8mD0gDIpww9
UXmc2ijdH7c6zqni4aZHVBHJW0dcB91lcRaH4Q7Obumz6HHbR/SOnP/FTDhCoROTF8wA+PdwnTM2KE1VYLNiKcA+mrEiflpcz0aY
aR1hpor7SMtdoRYoe5eWuqUwvhwBCMsT/6XNoqAgMbx4DKJAmooknKTLdA3baLRmyYojUyQ6WwKDCYv+L3b46sk=
""")).decode(), globals())


This notebook takes one classifier from a decision written on paper to a model running at a help desk. The case is the same as in the video: tickets about account takeover, where someone else has got into a customer's account, go to a small security team. Only 1 ticket in 100 is one of them. A model that says "other" for every ticket is 99% accurate and catches nothing, and by the end you'll have the tools to show, in code, why a model with lower accuracy is far better.

You'll work through the same ideas as the video, in code:

1. The task
2. The confusion matrix
3. Precision and recall
4. The threshold
5. Choosing by cost
6. Class imbalance
7. Small test counts
8. Leakage
9. Leaks in the split
10. Error analysis
11. Label agreement
12. Drift
13. Monitoring
14. The pipeline and the payoff

Every section starts with a short explanation and a worked example you can run. Then come exercises that build on each other. Work them in order.

The tickets here are synthetic: made up by a short program with a fixed seed, so everyone gets the same 20,000 tickets. The arithmetic exercises use the video's own numbers, so you can check them against what you saw.

In [ ]:
#@title Helpers, imports and the ticket data
import math

import matplotlib.pyplot as plt
import torch.nn.functional as F

torch.manual_seed(0)
RED, GREY = PALETTE["pink"], "#5a5290"

GENERAL = "please help my account i the can you thanks today need with is it hi".split()
WORDS = {
    "takeover": "hacked:2 stolen:1 someone:3 changed:3 password:4 email:3 login:3 locked:3 access:3 strange:2",
    "short": "hacked:1 locked:2 login:2 password:2 access:1",
    "purchase": "purchase:4 didn't:3 order:2 unknown:2 charge:3 recognise:2",
    "newscam": "code:4 text:4 verification:3 read:2 back:2 caller:2 package:2 delivery:2 app:2",
    "reset": "forgot:4 password:5 reset:4 link:3 login:3 email:2 locked:2 changed:1 new:2",
    "travel": "card:4 declined:4 travelling:3 abroad:2 payment:2 blocked:2 locked:1 strange:1",
    "phish": "email:4 real:3 received:3 link:3 suspicious:3 someone:2 strange:2 access:1",
    "billing": "charge:4 refund:4 invoice:2 bill:2 twice:2 payment:2 unknown:1 purchase:1",
    "shipping": "order:4 delivery:3 late:3 package:3 where:2 tracking:2",
    "app": "app:4 crash:3 slow:3 update:3 screen:2 login:2",
    "plan": "upgrade:3 plan:4 cancel:3 price:2 subscription:3",
}
WORDS = {k: ([w.split(":")[0] for w in v.split()], np.array([float(w.split(":")[1]) for w in v.split()])) for k, v in WORDS.items()}
OTHER_TOPICS = {"reset": 0.16, "travel": 0.10, "phish": 0.06, "billing": 0.20, "shipping": 0.20, "app": 0.16, "plan": 0.12}
REPLIES = {"reset": "sent you a new reset link", "travel": "card unblocked for your trip",
           "phish": "that message did not come from us", "billing": "refund is on its way",
           "shipping": "tracking number sent", "app": "please install the latest update", "plan": "your plan is changed"}
FORWARD = ["forwarded to the security team", "passed to security", "security team will call"]


def make_text(r, topic):
    """One synthetic ticket: a few words from its topic plus a few common words, in random order."""
    words, p = WORDS[topic]
    k = 1 if topic == "short" else int(r.integers(2, 5))
    ws = list(r.choice(words, k, replace=False, p=p / p.sum()))
    ws += list(r.choice(GENERAL, 1 if topic == "short" else r.integers(3, 6)))
    if topic in OTHER_TOPICS and r.random() < 0.05:
        ws.append(r.choice(WORDS["takeover"][0]))
    r.shuffle(ws)
    return " ".join(ws)


def make_year(seed=221):
    """12 months of tickets, sorted by day. Exactly 140, 30 and 30 security tickets fall in the three date blocks."""
    r = np.random.default_rng(seed)
    n = 20000
    day = np.sort(r.integers(0, 365, n))
    customer = r.integers(0, 15000, n)
    y = -np.ones(n, dtype=int)
    group = -np.ones(n, dtype=int)
    names = list(OTHER_TOPICS)
    p = np.array([OTHER_TOPICS[k] for k in names])
    topic = np.array(r.choice(names, n, p=p / p.sum()), dtype=object)
    g = 0
    for lo, hi, k in [(0, 14000, 140), (14000, 17000, 30), (17000, 20000, 30)]:
        done = 0
        while done < k:
            at = int(r.integers(lo, hi - 3))
            size = int(min(r.choice([1, 1, 1, 2, 3]), k - done))
            if np.any(y[at:at + size] == 1):
                continue
            y[at:at + size] = 1
            group[at:at + size] = g
            g += 1
            customer[at:at + size] = customer[at]
            day[at:at + size] = day[at]
            topic[at:at + size] = r.choice(["takeover", "short", "purchase"], p=[0.72, 0.14, 0.14])
            done += size
    text = np.empty(n, dtype=object)
    for i in range(n):
        if group[i] >= 0 and i > 0 and group[i - 1] == group[i]:
            ws = text[i - 1].split()          # a near-duplicate: the same ticket sent again, one word changed
            ws[r.integers(len(ws))] = r.choice(GENERAL)
            text[i] = " ".join(ws)
        else:
            text[i] = make_text(r, topic[i])
    reply = np.array([r.choice(FORWARD) if (y[i] == 1 and r.random() < 0.97) or (topic[i] == "phish" and r.random() < 0.03)
                      else REPLIES.get(topic[i], REPLIES["reset"]) for i in range(n)], dtype=object)
    ticket_id = np.where(y == 1, 900000, 100000) + np.arange(n)
    n_msg = np.where(y == 1, 3 + r.poisson(4, n), 1 + r.poisson(1.5, n))
    return day, customer, y, topic, text, reply, ticket_id, n_msg


def make_week(seed, n=3000, rate=0.01, new_share=0.0):
    """One week of live tickets. A share new_share of the security tickets use the new scam's wording."""
    r = np.random.default_rng(seed)
    n_pos = round(n * rate)
    names = list(OTHER_TOPICS)
    p = np.array([OTHER_TOPICS[k] for k in names])
    topic = list(r.choice(names, n - n_pos, p=p / p.sum()))
    n_new = round(n_pos * new_share)
    topic += ["newscam"] * n_new + list(r.choice(["takeover", "short", "purchase"], n_pos - n_new, p=[0.72, 0.14, 0.14]))
    topic = np.array(topic, dtype=object)
    y = np.where(np.isin(topic, ["takeover", "short", "purchase", "newscam"]), 1, -1)
    order = r.permutation(n)
    topic, y = topic[order], y[order]
    return np.array([make_text(r, t) for t in topic], dtype=object), y, topic


DAY, CUSTOMER, Y, TOPIC, TEXT, REPLY, TICKET_ID, N_MSG = make_year()
MONTH = np.minimum(DAY * 12 // 365, 11)
N_ALL = len(Y)


def draw_matrix(ax, cm, title=""):
    """A confusion matrix as a coloured 2 by 2 grid: rows are true labels, columns are predictions."""
    tp, fn, fp, tn = cm
    grid = np.array([[tp, fn], [fp, tn]], dtype=float)
    colours = np.array([[PALETTE["mint"], PALETTE["pink"]], [PALETTE["amber"], PALETTE["violet"]]])
    for i in range(2):
        for j in range(2):
            ax.add_patch(plt.Rectangle((j, 1 - i), 1, 1, color=colours[i, j], alpha=0.85))
            ax.text(j + 0.5, 1.5 - i, f"{grid[i, j]:,.0f}", ha="center", va="center", fontsize=15,
                    color=PALETTE["night"], fontweight="bold")
    ax.set_xlim(0, 2); ax.set_ylim(0, 2); ax.grid(False)
    ax.set_xticks([0.5, 1.5], ["flagged", "not flagged"]); ax.set_yticks([1.5, 0.5], ["security", "other"])
    if title:
        ax.set_title(title, fontsize=10)


def draw_strip(ax, scores, y, cut=None, title=""):
    """Scored tickets in a row, highest first: red for security, grey for other."""
    for k, (s, lab) in enumerate(zip(scores, y)):
        ax.add_patch(plt.Rectangle((k, 0), 0.85, 1, color=RED if lab == 1 else GREY))
        ax.text(k + 0.42, -0.35, f"{s:.2f}", ha="center", fontsize=7, rotation=90, color=PALETTE["ink"])
    if cut is not None:
        k = int(np.sum(np.asarray(scores) >= cut - 1e-12))
        ax.axvline(k - 0.07, color=PALETTE["amber"], lw=3)
    ax.set_xlim(-0.3, len(scores)); ax.set_ylim(-1, 1.3); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)

Here are the opening's 3,000 test tickets as dots: 30 of them, in red, are real takeovers. A model that says "other" for every ticket gets the 2,970 grey dots right and every red dot wrong. That is 99% accuracy and 0 catches.

In [ ]:
r_demo = np.random.default_rng(7)
cells = np.zeros(3000)
cells[r_demo.choice(3000, 30, replace=False)] = 1
fig, ax = plt.subplots(figsize=(10, 4.2))
gx, gy = np.arange(3000) % 75, np.arange(3000) // 75
ax.scatter(gx[cells == 0], gy[cells == 0], s=6, color=GREY)
ax.scatter(gx[cells == 1], gy[cells == 1], s=40, color=RED)
ax.set_title("3,000 test tickets, 30 takeovers. Always saying 'other': 99% accurate, 0 of 30 caught")
ax.axis("off")
plt.show()



## <font color="#ff2e88">The task</font>

The first step in building a classifier is to write down the decision it supports. Every new ticket either goes to the security team, who promise to look within 15 minutes, or stays in the main queue, where a reply may take a day.

So the classifier has two labels. $+1$ means **security**: the ticket is about an account takeover. $-1$ means **other**: everything else. The label we are trying to find, the rare and important one, is called the **positive class**. Here the positive class is security.

Next, write down who acts on each prediction, and put a price on each kind of mistake. A missed takeover leaves a customer locked out while someone else spends their money, and the help desk charges it 50. A needless check, where the security team opens a ticket that turns out to be a password question, costs a few minutes and is charged 1.

The ratio of the two, 50 to 1, is the **cost ratio**. One missed takeover is as bad as 50 needless checks. These numbers come from the people who run the help desk, before anyone trains a model, and they decide how every model is judged.

Last, measure how common the positive class is. In 12 months, 200 of 20,000 tickets were about account takeover. That share, 1%, is the **base rate**.

The data is 20,000 labelled tickets, each with its text and its label. The model is the one from the lesson on classification: word count features, a weight for each, the sigmoid, and the logistic loss. In code, a ticket's text becomes a vector of word counts over a fixed list of words, the **vocabulary**, built from the training tickets.

The data cell above made the arrays `TEXT`, `Y`, `DAY`, `MONTH`, `CUSTOMER` and a few more, one entry per ticket, sorted by day. Run the cell below to see some tickets and the base rate.

In [ ]:
C_MISS, C_FA = 50, 1
for i in [5, 40, int(np.flatnonzero(Y == 1)[0]), 77, int(np.flatnonzero(Y == 1)[3])]:
    print(f"{'security' if Y[i] == 1 else 'other   '}  day {DAY[i]:3}  {TEXT[i]}")
print(f"\n{N_ALL:,} tickets, {np.sum(Y == 1)} security, base rate {np.mean(Y == 1):.1%}")

fig, ax = plt.subplots(figsize=(9, 1.4))
ax.barh([0], [np.mean(Y == 1)], color=RED)
ax.barh([0], [np.mean(Y == -1)], left=[np.mean(Y == 1)], color=GREY)
ax.set_xlim(0, 1); ax.set_yticks([]); ax.set_title("The base rate: a thin red slice")
plt.show()



<font size="4"><b>The cost ratio</b></font>

Store `cost_ratio`, how many needless checks cost as much as one missed takeover, from `C_MISS` and `C_FA`.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Use the 2 prices set above. The ratio says how many false alarms cost the same as one miss.

</details>

In [ ]:
# Exercise 1.1: running this cell checks your answer
cost_ratio = ...  # your answer



<font size="4"><b>The base rate</b></font>

Store `base_rate`, the share of all tickets in `Y` that are security tickets.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

The mean of a boolean array is the share of its values that are True.

</details>

In [ ]:
# Exercise 1.2: running this cell checks your answer
base_rate = ...  # your answer



<font size="4"><b>A busy month</b></font>

A busy month brings 30,000 tickets at the same base rate. Store `busy_pos`, the number of security tickets you expect, and `per_day`, that number spread over 30 days.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

The base rate is the share of tickets that are security. Spread the month's total over its days.

</details>

In [ ]:
# Exercise 1.3: running this cell checks your answer
busy_pos = ...  # your answer
per_day = ...  # your answer



<font size="4"><b>Labels from team names</b></font>

The help desk's export names the team that handled each ticket. Write `to_label(teams)` that turns a list of strings `"security"` or `"other"` into a NumPy array of $+1$ and $-1$.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

`np.where(condition, 1, -1)` picks between two values element by element.

</details>

In [ ]:
# Exercise 1.4: running this cell checks your answer
def to_label(teams: list[str]) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Tokenize</b></font>

Write `tokenize(text)`: the list of words in a ticket, lower case, split on white space.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Python strings have a method for lowercase and a method that splits on whitespace.

</details>

In [ ]:
# Exercise 1.5: running this cell checks your answer
def tokenize(text: str) -> list[str]:
    return ...  # your code here



<font size="4"><b>Count vector</b></font>

Write `count_vector(text, vocab)`: a NumPy array of length `len(vocab)` whose entry `j` counts how often `vocab[j]` appears in the ticket. Words outside the vocabulary are ignored.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Build a dictionary from word to position with `enumerate(vocab)`, then add 1 at the position of each known word from your `tokenize`.

</details>

In [ ]:
# Exercise 1.6: running this cell checks your answer
def count_vector(text: str, vocab: list[str]) -> np.ndarray:
    ...  # your code here



<font size="4"><b>Security tickets per month</b></font>

Store `pos_per_month`, an array of 12 counts: how many security tickets arrived in each month, using `MONTH` (0 to 11). A bar chart appears when the check passes.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

`np.bincount` counts how often each integer appears. Pass `minlength=12` so a month with none still gets a 0.

</details>

In [ ]:
# Exercise 1.7: running this cell checks your answer
pos_per_month = ...  # your answer



<font size="4"><b>The price of a set of mistakes</b></font>

Write `total_cost(n_miss, n_fa, c_miss=50, c_fa=1)`: the total charge for `n_miss` missed takeovers and `n_fa` needless checks.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Each kind of mistake has its own price, and the total covers both kinds.

</details>

In [ ]:
# Exercise 1.8: running this cell checks your answer
def total_cost(n_miss: int, n_fa: int, c_miss: float = 50, c_fa: float = 1) -> float:
    return ...  # your code here



## <font color="#7b2ff7">The confusion matrix</font>

Hold back 3,000 test tickets: 30 security and 2,970 other. Each ticket has a true label, and the model gives each one a prediction. There are 4 ways a prediction can land:

* a security ticket the model flags is a **true positive** (TP);
* a security ticket the model misses is a **false negative** (FN);
* an other ticket the model flags is a **false positive** (FP), also called a **false alarm**;
* an other ticket the model leaves alone is a **true negative** (TN).

The second word is what the model said: positive means it flagged the ticket. The first word says whether it was right.

A **confusion matrix** counts all 4 in a 2 by 2 grid. Rows are the true labels, security on top. Columns are the predictions, flagged on the left.

**Accuracy** is the share of tickets on the diagonal, where the prediction matches the label:
$$\text{accuracy} = \frac{TP + TN}{TP + FN + FP + TN}.$$

The model that always says other never flags, so its left column is empty: 30 false negatives and 2,970 true negatives, an accuracy of 99%. The main model, the logistic classifier, flags 24 security tickets, misses 6, raises 120 false alarms and leaves 2,850 alone. Its accuracy is 95.8%, lower than the model that does nothing.

Accuracy counts every ticket the same. When one class is 99% of the data, accuracy mostly measures the common class. The matrix keeps the 4 counts apart, so nothing hides.

In this notebook a confusion matrix is the tuple `(tp, fn, fp, tn)`, in that order. Labels and predictions are arrays of $+1$ and $-1$.

Below, the data is split by date (section 9 explains why): the first 14,000 tickets for training, the next 3,000 for validation, the last 3,000 for test. The main model is trained on the training tickets with a class weight (section 6 explains that too), and `P_VA` and `P_TE` hold its probability for each validation and test ticket.

In [ ]:
MAIN = (24, 6, 120, 2850)       # the video's main model on its test set
BASE = (0, 30, 0, 2970)         # always "other"
TR, VA, TE = slice(0, 14000), slice(14000, 17000), slice(17000, 20000)
VOCAB = sorted({w for t in TEXT[TR] for w in _tokens(t)})
X_ALL = _counts(TEXT, VOCAB)
X_TR, X_VA, X_TE = X_ALL[TR], X_ALL[VA], X_ALL[TE]
Y_TR, Y_VA, Y_TE = Y[TR], Y[VA], Y[TE]
W_MAIN, B_MAIN = _train(X_TR, Y_TR, w_pos=99.0)
P_VA, P_TE = _prob(X_VA, W_MAIN, B_MAIN), _prob(X_TE, W_MAIN, B_MAIN)
print(f"{len(VOCAB)} words in the vocabulary; security tickets: train {np.sum(Y_TR == 1)}, validation {np.sum(Y_VA == 1)}, test {np.sum(Y_TE == 1)}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
draw_matrix(axes[0], BASE, "Always other: accuracy 99%")
draw_matrix(axes[1], MAIN, "Main model: accuracy 95.8%")
plt.tight_layout()
plt.show()



<font size="4"><b>Count the four cells</b></font>

Write `confusion(y, pred)` returning `(tp, fn, fp, tn)` as plain integers, for arrays of $+1$ and $-1$.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Combine 2 boolean masks with `&`, one on the label and one on the prediction, and count the True values. Do this once for each cell.

</details>

In [ ]:
# Exercise 2.1: running this cell checks your answer
def confusion(y, pred) -> tuple[int, int, int, int]:
    ...  # your code here



<font size="4"><b>Ten tickets</b></font>

`TEN` lists 10 tickets as (true label, prediction) pairs. Store `ten`, their confusion matrix.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Split the pairs into two arrays first: `np.array(TEN)[:, 0]` is the labels.

</details>

In [ ]:
# Exercise 2.2: running this cell checks your answer
TEN = [(1, 1), (1, -1), (1, 1), (-1, 1), (-1, -1), (-1, -1), (-1, -1), (-1, 1), (-1, -1), (-1, -1)]
ten = ...  # your answer



<font size="4"><b>Accuracy</b></font>

Write `accuracy(tp, fn, fp, tn)`: the share of tickets on the diagonal.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The diagonal is the true positives and the true negatives.

</details>

In [ ]:
# Exercise 2.3: running this cell checks your answer
def accuracy(tp, fn, fp, tn) -> float:
    return ...  # your code here



<font size="4"><b>Always other, on our test split</b></font>

Our synthetic test split also holds 30 security tickets. Store `base_cm`, the confusion matrix of predicting $-1$ for every ticket in `Y_TE`, and `base_acc`, its accuracy.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`-np.ones_like(Y_TE)` is a prediction of other for every ticket.

</details>

In [ ]:
# Exercise 2.4: running this cell checks your answer
base_cm = ...  # your answer
base_acc = ...  # your answer



<font size="4"><b>The main model&#x27;s accuracy</b></font>

Store `main_acc`, the accuracy of the video's main model `MAIN`, and `more_accurate`, the string `"always other"` or `"main"` for the model with the higher accuracy.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Your `accuracy` takes the 4 counts, and `*` unpacks a tuple into arguments.

</details>

In [ ]:
# Exercise 2.5: running this cell checks your answer
main_acc = ...  # your answer
more_accurate = ...  # your answer



<font size="4"><b>The matrix as a grid</b></font>

Write `as_grid(tp, fn, fp, tn)`: a 2 by 2 NumPy array with rows (security, other) and columns (flagged, not flagged). A picture of `MAIN` appears when the check passes.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Rows are the true label and columns are the prediction, security first in both.

</details>

In [ ]:
# Exercise 2.6: running this cell checks your answer
def as_grid(tp, fn, fp, tn) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Find the bug</b></font>

The function below has a bug. Store `swapped`, the set of the two count names (from `"tp"`, `"fn"`, `"fp"`, `"tn"`) that come out in each other's place.

```python
def confusion_bug(y, pred):
    tp = np.sum((y == 1) & (pred == 1))
    fn = np.sum((y == -1) & (pred == 1))
    fp = np.sum((y == 1) & (pred == -1))
    tn = np.sum((y == -1) & (pred == -1))
    return tp, fn, fp, tn
```

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

A false negative is a security ticket that was missed: what should its label and its prediction be?

</details>

In [ ]:
# Exercise 2.7: running this cell checks your answer
swapped = ...  # your answer



<font size="4"><b>Our model on validation</b></font>

Store `cm_va`, the confusion matrix of the trained model on the validation tickets, flagging every ticket with `P_VA` at or above 0.5, and `acc_va`, its accuracy.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`np.where(P_VA >= 0.5, 1, -1)` gives the predictions.

</details>

In [ ]:
# Exercise 2.8: running this cell checks your answer
cm_va = ...  # your answer
acc_va = ...  # your answer



## <font color="#00a383">Precision and recall</font>

Two ratios read the matrix one line at a time.

**Recall** is the share of real security tickets that the model catches. It reads the top row:
$$\text{recall} = \frac{TP}{TP + FN}.$$
The main model caught 24 of 30, a recall of 80%.

**Precision** is the share of flagged tickets that are real. It reads the left column:
$$\text{precision} = \frac{TP}{TP + FP}.$$
The main model flagged 144 tickets and 24 were real, a precision of about 16.7%. About 1 flag in 6 is a takeover.

Both share one cell, the true positives. The bottom right cell, the 2,850 true negatives, appears in neither, so easy correct answers on the common class never enter precision or recall.

Each answers a different person's question. The customer asks about recall: of the takeovers that happened, how many got help in time? The security team asks about precision: of the tickets in our tray, how many deserve our time?

The model that always says other has recall 0, and its precision cannot be computed, because it flagged nothing. In code we return `None` for that case.

The two pull against each other. Flag every ticket and recall is 100% while precision drops to the base rate. Flag only the single most certain ticket and precision may be perfect while recall is near 0. So a model is judged on the pair, reported with the matrix they come from.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
draw_matrix(axes[0], MAIN, "Recall reads the top row: 24 of 30")
axes[0].add_patch(plt.Rectangle((0, 1), 2, 1, fill=False, ec=PALETTE["cyan"], lw=4))
draw_matrix(axes[1], MAIN, "Precision reads the left column: 24 of 144")
axes[1].add_patch(plt.Rectangle((0, 0), 1, 2, fill=False, ec=PALETTE["cyan"], lw=4))
plt.tight_layout()
plt.show()



<font size="4"><b>Precision</b></font>

Write `precision(tp, fp)`, returning `None` when nothing was flagged.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Check `tp + fp == 0` first.

</details>

In [ ]:
# Exercise 3.1: running this cell checks your answer
def precision(tp, fp) -> float | None:
    return ...  # your code here



<font size="4"><b>Recall</b></font>

Write `recall(tp, fn)`, returning `None` when there are no security tickets at all.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

The bottom of the fraction is the whole top row of the matrix. Guard against 0 as in `precision`.

</details>

In [ ]:
# Exercise 3.2: running this cell checks your answer
def recall(tp, fn) -> float | None:
    return ...  # your code here



<font size="4"><b>The main model&#x27;s pair</b></font>

Store `p_main` and `r_main` for `MAIN`, and `flags_per_catch`, how many flagged tickets the security team opens per real takeover.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`MAIN` is `(tp, fn, fp, tn)`. Flags per catch is the flagged count over the true positives.

</details>

In [ ]:
# Exercise 3.3: running this cell checks your answer
p_main = ...  # your answer
r_main = ...  # your answer
flags_per_catch = ...  # your answer



<font size="4"><b>A second model</b></font>

A second model catches 15 security tickets, misses 15 and raises 15 false alarms on the same 3,000. Store `p2` and `r2`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Both have 15 on top.

</details>

In [ ]:
# Exercise 3.4: running this cell checks your answer
p2 = ...  # your answer
r2 = ...  # your answer



<font size="4"><b>Always other</b></font>

Store `base_p` and `base_r`, the precision and recall of the always-other model `BASE`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Use your two functions. One of them has nothing to divide.

</details>

In [ ]:
# Exercise 3.5: running this cell checks your answer
base_p = ...  # your answer
base_r = ...  # your answer



<font size="4"><b>Flag everything</b></font>

Flag every ticket in the test split. Store `all_p` and `all_r` for that rule, computed from `Y_TE` with your `confusion`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`np.ones_like(Y_TE)` flags everything.

</details>

In [ ]:
# Exercise 3.6: running this cell checks your answer
cm_all = ...  # your answer
all_p = ...  # your answer
all_r = ...  # your answer



<font size="4"><b>Flag only the top ticket</b></font>

Flag only the single validation ticket with the highest `P_VA`. Store `top_p` and `top_r`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Start from a prediction of other for every ticket, then flag the position of the highest probability. `np.argmax` finds it.

</details>

In [ ]:
# Exercise 3.7: running this cell checks your answer
top_p, top_r = ..., ...  # your answer



<font size="4"><b>True negatives never enter</b></font>

Predict, then verify. Make `MAIN_BIG`, the main model's counts with 10 times as many true negatives. Store `acc_big`, its accuracy, and `pr_big`, its (precision, recall).

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Only the last count changes: `MAIN[3] * 10`.

</details>

In [ ]:
# Exercise 3.8: running this cell checks your answer
MAIN_BIG = ...  # your answer
acc_big = ...  # your answer
pr_big = ...  # your answer



## <font color="#ff5f2e">The threshold</font>

The logistic classifier computes a score for each ticket, and the sigmoid turns it into a number between 0 and 1: the model's estimate of the chance that the ticket is about security.

The **threshold** is the cut-off we choose. Tickets with a probability at or above it are flagged. 0.5 is the usual default, and nothing makes it right for this problem.

Take 20 validation tickets sorted by probability, highest first. 5 of them, in red, are security tickets. At a cut of 0.75, 3 are flagged and 2 are red: precision 2 in 3, recall 2 in 5. Slide the cut to 0.5 and 7 are flagged, 4 of them red: precision about 57%, recall 80%. At 0.2, 12 are flagged including all 5 red ones: recall 100%, precision about 42%.

Lowering the threshold flags more tickets. Recall can only go up or stay put, because no flagged ticket is ever unflagged. Precision usually goes down, because the tickets added near the cut are the doubtful ones.

Plot each threshold as a point, recall across and precision up, and join the points. That is a **precision-recall curve**: every trade the model offers. Use every ticket's probability as a threshold in turn and the curve fills in, zigzagging as red and grey tickets join.

Training chose the weights and with them the ranking. The threshold only chooses where to cut that ranking, so one trained model gives a whole curve of classifiers. A better model has a curve that sits higher.

In [ ]:
STRIP = np.array([0.95, 0.90, 0.80, 0.70, 0.60, 0.55, 0.50, 0.40, 0.35, 0.30, 0.25, 0.20, 0.15, 0.12, 0.10, 0.08, 0.06, 0.05, 0.03, 0.02])
STRIP_Y = np.where(np.isin(STRIP, [0.95, 0.90, 0.70, 0.50, 0.25]), 1, -1)
fig, axes = plt.subplots(3, 1, figsize=(10, 4.6))
for ax, t in zip(axes, [0.75, 0.5, 0.2]):
    draw_strip(ax, STRIP, STRIP_Y, cut=t, title=f"cut at {t}: everything left of the amber line is flagged")
plt.tight_layout()
plt.show()



<font size="4"><b>Flag at a threshold</b></font>

Write `flag(probs, t)`: an array of $+1$ where a probability is at or above `t`, and $-1$ elsewhere.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`np.where(condition, 1, -1)`. The threshold itself counts as flagged.

</details>

In [ ]:
# Exercise 4.1: running this cell checks your answer
def flag(probs, t: float) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>The strip at 0.75</b></font>

Store `cm75`, the strip's confusion matrix at a threshold of 0.75, and `pr75`, its (precision, recall).

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Your `flag` and `confusion`, then your two ratios.

</details>

In [ ]:
# Exercise 4.2: running this cell checks your answer
cm75 = ...  # your answer
pr75 = ...  # your answer



<font size="4"><b>The strip at 0.35</b></font>

Store `pr35`, the strip's (precision, recall) at a threshold of 0.35.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Count the flagged tickets first, then the red ones among them.

</details>

In [ ]:
# Exercise 4.3: running this cell checks your answer
cm35 = ...  # your answer
pr35 = ...  # your answer



<font size="4"><b>Sweep the thresholds</b></font>

Write `sweep(probs, y, thresholds)` returning two NumPy arrays, precision and recall at each threshold. Use `np.nan` where precision has no flags to divide.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Loop over the thresholds; for each, build the matrix with `confusion(y, flag(probs, t))`. Replace a `None` from `precision` with `np.nan`.

</details>

In [ ]:
# Exercise 4.4: running this cell checks your answer
def sweep(probs, y, thresholds):
    ...  # your code here



<font size="4"><b>Recall never rises</b></font>

Sweep the validation probabilities `P_VA` over the thresholds `T_GRID`, from 0 up to 1. Store `rec_grid`, the recall at each, and `recall_never_rises`, whether recall never goes up as the threshold goes up.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`np.diff(rec_grid)` holds the changes from one threshold to the next.

</details>

In [ ]:
# Exercise 4.5: running this cell checks your answer
T_GRID = np.linspace(0, 1, 101)
rec_grid = ...  # your answer
recall_never_rises = ...  # your answer



<font size="4"><b>Every point on the curve</b></font>

Write `pr_points(probs, y)`: sort the tickets by probability, highest first, and flag the top 1, then the top 2, and so on. Return two arrays, recall and precision, one entry per cut. The strip's curve appears when the check passes.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Sort the tickets by probability, highest first, with `np.argsort`. A running count of security tickets down that order (`np.cumsum`) gives the catches in the top `k`, for every `k` at once.

</details>

In [ ]:
# Exercise 4.6: running this cell checks your answer
def pr_points(probs, y):
    ...  # your code here



<font size="4"><b>Only the ranking matters</b></font>

Cubing every probability changes the numbers and keeps their order. Store `curve_gap`, the largest absolute difference between the precisions from `pr_points(P_VA, Y_VA)` and from `pr_points(P_VA ** 3, Y_VA)`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Take index 1 of each result, subtract, then `np.abs(...).max()`.

</details>

In [ ]:
# Exercise 4.7: running this cell checks your answer
curve_gap = ...  # your answer



<font size="4"><b>Precision at 80% recall</b></font>

Store `prec_at_80`, the highest precision any cut reaches on the validation tickets while keeping recall at 80% or more. The curve appears when the check passes.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Unpack the 2 arrays from `pr_points`, keep the points with recall at least 0.8, and take the best precision among them.

</details>

In [ ]:
# Exercise 4.8: running this cell checks your answer
R_va, P_va = ..., ...  # your answer
prec_at_80 = ...  # your answer



## <font color="#d0309f">Choosing by cost</font>

The costs from section 1 pick the threshold. For each threshold, count the mistakes on the validation set and price them: every miss costs 50, every false alarm 1. The best threshold is the one with the lowest total cost.

On the strip: at 0.75, 3 misses and 1 false alarm cost 151. At 0.5, 1 miss and 3 false alarms cost 53. At 0.2, no misses and 7 false alarms cost 7. With misses 50 times as expensive, the team should flag eagerly.

Change the costs and the answer can change. If a miss cost only twice a false alarm, the totals would be 7, 5 and 7, and 0.5 would win. The cut belongs to the people who pay for the mistakes.

On the test set, the always-other model misses all 30, for a cost of 1,500. The main model misses 6 and raises 120 false alarms, for 420. The model with lower accuracy costs less than a third as much.

When nobody can name the costs, a common fallback is the **F1 score**, the harmonic mean of precision and recall:
$$F_1 = \frac{2PR}{P + R}.$$
It stays low unless both are reasonably high. On the strip F1 is highest at 0.5, about 0.67, which is the wrong cut when a miss costs 50. Use F1 only when the costs are truly unknown.

Choose the threshold on the validation set, never on the test set. The threshold is a setting, like $\lambda$ in the last lesson, and the test set is still opened once, at the end.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
for k, (t, miss, fa) in enumerate([(0.75, 3, 1), (0.5, 1, 3), (0.2, 0, 7)]):
    ax.bar(k, miss * 50, color=RED, label="misses × 50" if k == 0 else None)
    ax.bar(k, fa, bottom=miss * 50, color=PALETTE["amber"], label="false alarms × 1" if k == 0 else None)
    ax.text(k, miss * 50 + fa + 4, str(miss * 50 + fa), ha="center")
ax.set_xticks(range(3), ["0.75", "0.5", "0.2"]); ax.set_xlabel("threshold"); ax.set_ylabel("total cost")
ax.legend(fontsize=8)
plt.show()



<font size="4"><b>Cost at a threshold</b></font>

Write `cost_at(probs, y, t, c_miss=50, c_fa=1)`: the total cost of the misses and false alarms when flagging at `t`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Build the matrix at threshold `t` with your `flag` and `confusion`, then let your `total_cost` price it.

</details>

In [ ]:
# Exercise 5.1: running this cell checks your answer
def cost_at(probs, y, t: float, c_miss: float = 50, c_fa: float = 1) -> float:
    tp, fn, fp, tn = ..., ..., ..., ...  # your answer
    return ...  # your code here



<font size="4"><b>The strip&#x27;s best cut at 50 to 1</b></font>

Store `strip_costs`, the costs at thresholds 0.75, 0.5 and 0.2 as a list, and `best50`, the threshold with the lowest cost.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

A list comprehension over `[0.75, 0.5, 0.2]`, then the threshold at the position of `np.argmin`.

</details>

In [ ]:
# Exercise 5.2: running this cell checks your answer
CUTS = [0.75, 0.5, 0.2]
strip_costs = ...  # your answer
best50 = ...  # your answer



<font size="4"><b>At 2 to 1</b></font>

Now a miss costs 2 and a false alarm 1. Store `strip_costs_2` and `best2`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Pass `c_miss=2`.

</details>

In [ ]:
# Exercise 5.3: running this cell checks your answer
strip_costs_2 = ...  # your answer
best2 = ...  # your answer



<font size="4"><b>The strip at 0.35, priced</b></font>

Store `cost35`, the strip's cost at 0.35 with prices 50 and 1.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

1 miss and 5 false alarms.

</details>

In [ ]:
# Exercise 5.4: running this cell checks your answer
cost35 = ...  # your answer



<font size="4"><b>The two models on the test set</b></font>

Store `base_cost` and `main_cost`, the costs of `BASE` and `MAIN`, and `cost_share`, the main model's cost as a fraction of the baseline's.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`total_cost(fn, fp)` with the counts in positions 1 and 2.

</details>

In [ ]:
# Exercise 5.5: running this cell checks your answer
base_cost = ...  # your answer
main_cost = ...  # your answer
cost_share = ...  # your answer



<font size="4"><b>The F1 score</b></font>

Write `f1(p, r)`, returning 0.0 when either is 0 or `None`. Store `f1_main` for the main model and `f1_strip`, the F1 at each of `CUTS` on the strip.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

F1 is the harmonic mean of precision and recall. Return 0 when either one is 0 or `None`; `not` catches both.

</details>

In [ ]:
# Exercise 5.6: running this cell checks your answer
def f1(p, r) -> float:
    return ...  # your code here


f1_main = ...  # your answer
f1_strip = ...  # your answer



<font size="4"><b>Pick our threshold on validation</b></font>

Store `costs_va`, the validation cost of our model at each threshold in `T_PICK`, and `t_star`, the threshold with the lowest cost (the first one if there is a tie). A plot appears when the check passes.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Price each threshold in `T_PICK` with your `cost_at`. `np.argmin` gives the position of the smallest cost.

</details>

In [ ]:
# Exercise 5.7: running this cell checks your answer
T_PICK = np.round(np.linspace(0.05, 0.99, 95), 2)
costs_va = ...  # your answer
t_star = ...  # your answer



<font size="4"><b>The break-even probability</b></font>

Suppose a model's probabilities were honest. Flagging a ticket with probability $p$ risks a false alarm with chance $1 - p$; leaving it risks a miss with chance $p$. Store `p_break`, the probability where the two expected costs are equal, from `C_MISS` and `C_FA`, and `flag_at_3pct`, whether a ticket with a 3% chance should be flagged.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Solve $p \cdot 50 = (1 - p) \cdot 1$ for $p$.

</details>

In [ ]:
# Exercise 5.8: running this cell checks your answer
p_break = ...  # your answer
flag_at_3pct = ...  # your answer



## <font color="#0077ff">Class imbalance</font>

The training set has 14,000 tickets and only 140 of them are security. When one class is much rarer than the other, the data has **class imbalance**.

The training loss is an average over every example. If each ticket starts with about the same loss, 99% of the total comes from other tickets and 1% from security tickets. Gradient descent reduces whichever part is biggest, and the quickest way to lower this loss is to push every score toward other.

The model that always predicts the most common class is the **majority baseline**. Any model you train must beat it on recall, precision and cost. If your matrix looks like the baseline's, the imbalance won.

The first fix is a **class weight**: multiply the loss of each security ticket by a number bigger than 1. Set it to the ratio of the counts, $13{,}860 / 140 = 99$, and the two halves of the loss match, 50% each. The weighted loss is
$$L(w) = \frac{1}{n}\sum_i c_i \log\big(1 + e^{-y_i s_i}\big), \qquad c_i = \begin{cases} 99 & y_i = +1 \\ 1 & y_i = -1. \end{cases}$$

One side effect: with the weight, the sigmoid's outputs are no longer honest probabilities. The model acts as if security were half the data. That is fine here, because the threshold is chosen on validation data anyway.

The second fix is **resampling**: show each security ticket several times per pass, or use only a fraction of the other tickets. The third, often the best, is to collect more real positives.

Whatever you change in training, test at the natural rate. The validation and test sets keep their 1%, because that is the mix the model will meet.

In PyTorch the weighted logistic loss is built in: `F.binary_cross_entropy_with_logits(s, t, pos_weight=...)`, with targets `t` of 1 and 0 in place of $+1$ and $-1$.

In [ ]:
n_pos, n_neg = int(np.sum(Y_TR == 1)), int(np.sum(Y_TR == -1))
fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
for ax, wpos, title in [(axes[0], 1, "Each ticket weighted 1"), (axes[1], 99, "Security tickets weighted 99")]:
    tot = n_neg + wpos * n_pos
    ax.barh([0], [wpos * n_pos / tot], color=RED)
    ax.barh([0], [n_neg / tot], left=[wpos * n_pos / tot], color=GREY)
    ax.set_xlim(0, 1); ax.set_yticks([]); ax.set_title(f"{title}: security share of the loss {wpos * n_pos / tot:.0%}")
plt.tight_layout()
plt.show()



<font size="4"><b>The class weight</b></font>

Store `class_weight`, the ratio of other tickets to security tickets in `Y_TR`, and `shares_after`, the (other, security) shares of the total weight once every security ticket counts `class_weight` times.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Count each class with `np.sum`. After weighting, the security part is `n_pos * class_weight`.

</details>

In [ ]:
# Exercise 6.1: running this cell checks your answer
class_weight = ...  # your answer
shares_after = ...  # your answer



<font size="4"><b>A smaller training set</b></font>

A smaller training set has 4,900 other tickets and 100 security tickets. Store `small_weight`, the class weight that balances them.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Divide the counts.

</details>

In [ ]:
# Exercise 6.2: running this cell checks your answer
small_weight = ...  # your answer



<font size="4"><b>The weighted loss</b></font>

Write `weighted_loss(s, y, w_pos)` in PyTorch: the mean over tickets of $c_i \log(1 + e^{-y_i s_i})$, where $c_i$ is `w_pos` for security tickets and 1 for the rest. `s` and `y` are float64 tensors.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`F.softplus(z)` is $\log(1 + e^z)$, computed safely. Build the weights with `torch.where(y > 0, w_pos, 1.0)`.

</details>

In [ ]:
# Exercise 6.3: running this cell checks your answer
def weighted_loss(s: torch.Tensor, y: torch.Tensor, w_pos: float) -> torch.Tensor:
    c = ...  # your answer
    return ...  # your code here



<font size="4"><b>Same as the built-in</b></font>

Store `builtin_loss`, the value of `F.binary_cross_entropy_with_logits` on `S_DEMO` and `Y_DEMO` with `pos_weight` 99, and `mine_loss`, your `weighted_loss` on the same, both as floats.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

The built-in wants targets of 1 and 0, so map -1 to 0 first. Its `pos_weight` argument takes a tensor.

</details>

In [ ]:
# Exercise 6.4: running this cell checks your answer
S_DEMO = torch.tensor(np.random.default_rng(6).normal(0, 2, 40))
Y_DEMO = torch.tensor(np.where(np.arange(40) < 4, 1.0, -1.0))
builtin_loss = ...  # your answer
mine_loss = ...  # your answer



<font size="4"><b>Train with a weight</b></font>

Write `train(X, y, w_pos, steps=300, lr=0.1)`: start the weights `w` and the bias `b` at 0 as float64 tensors, then take `steps` steps of `torch.optim.Adam([w, b], lr=lr)` on your `weighted_loss` of the scores `X @ w + b`. Return `w` as a NumPy array and `b` as a float.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`w = torch.zeros(X.shape[1], dtype=torch.float64, requires_grad=True)` and `b = torch.zeros((), dtype=torch.float64, requires_grad=True)`. Each step: `opt.zero_grad()`, compute the loss, `.backward()`, `opt.step()`.

</details>

In [ ]:
# Exercise 6.5: running this cell checks your answer
def train(X: np.ndarray, y: np.ndarray, w_pos: float, steps: int = 300, lr: float = 0.1):
    ...  # your code here



<font size="4"><b>Weight 1 against weight 99</b></font>

Train two models on the full training split with your `train`: one with `w_pos=1` and one with `w_pos=99`. Store `rec_w1` and `rec_w99`, the validation recall of each at a threshold of 0.5.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`_prob(X_VA, w, b)` turns weights into probabilities. Then `confusion` and `recall`.

</details>

In [ ]:
# Exercise 6.6: running this cell checks your answer
w1, b1 = ..., ...  # your answer
w99, b99 = ..., ...  # your answer
rec_w1 = ...  # your answer
rec_w99 = ...  # your answer



<font size="4"><b>Probabilities after weighting</b></font>

Store `mean_p_va`, the mean of the weighted model's probabilities `P_VA`, and `share_va`, the real share of security tickets in validation.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Two means.

</details>

In [ ]:
# Exercise 6.7: running this cell checks your answer
mean_p_va = ...  # your answer
share_va = ...  # your answer



<font size="4"><b>Resampling</b></font>

Build `resample_idx`, an index array into the training split holding every other ticket once and every security ticket 99 times. Store `resampled_share`, the share of security tickets in `Y_TR[resample_idx]`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`np.flatnonzero` gives the positions of each class, `np.repeat` copies each position, and `np.concatenate` joins arrays.

</details>

In [ ]:
# Exercise 6.8: running this cell checks your answer
resample_idx = ...  # your answer
resampled_share = ...  # your answer



<font size="4"><b>A balanced test set flatters</b></font>

Keep the main model's two rates, recall 0.8 and false alarm rate 120 / 2,970, and test it on a balanced set with as many other tickets as security tickets. Store `balanced_prec`, the precision there, and `natural_prec`, the precision at the natural rate (from `MAIN`).

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

On a balanced set of $m$ security and $m$ other tickets, there are $0.8m$ true positives and $(120/2970)m$ false alarms.

</details>

In [ ]:
# Exercise 6.9: running this cell checks your answer
balanced_prec = ...  # your answer
natural_prec = ...  # your answer



## <font color="#7b2ff7">Small test counts</font>

The last lesson split 1,000 labelled examples into 700 for training, 150 for validation and 150 for test. That works when the classes are balanced.

At a 1% base rate, 150 test tickets hold about 1.5 security tickets on average, and some draws hold none. Say the test set holds 2. Then recall can take only 3 values: 0, 50% or 100%. One ticket swings it by 50 points.

With 30 security tickets, each moves recall by about 3.3 points. That is still coarse, and it is enough to tell 80% from 50%.

Luck still plays a part at 30. A model whose true recall is 80% catches anywhere from 20 to 28 of the 30 about 96 times in 100. The count of catches follows a **binomial distribution**: the number of successes in $n$ independent tries that each succeed with chance $p$,
$$P(k) = \binom{n}{k} p^k (1 - p)^{n - k}.$$

So size the test set by the number of positives it holds. A rough rule is at least a few dozen of the rare class in validation and in test. At 1%, that means thousands of tickets each, which is why this case holds back 6,000.

Labelled data is expensive. Cross-validation, from the last lesson, helps here too: every security ticket gets a turn in validation. With a rare class, split the folds so each gets its share of positives; that is called a **stratified** split.

In [ ]:
rng_small = np.random.default_rng(0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
pos_in_150 = rng_small.binomial(150, 0.01, 10000)
axes[0].bar(*np.unique(pos_in_150, return_counts=True), color=PALETTE["violet"])
axes[0].set_title("Security tickets in a 150-ticket test set"); axes[0].set_xlabel("count")
k = np.arange(31)
axes[1].bar(k / 30, [math.comb(30, j) * 0.8 ** j * 0.2 ** (30 - j) for j in k], width=0.025, color=PALETTE["cyan"])
axes[1].set_title("Measured recall on 30 positives, true recall 80%"); axes[1].set_xlabel("measured recall")
plt.tight_layout()
plt.show()



<font size="4"><b>A test set of 150</b></font>

Store `expected_150`, the expected number of security tickets in 150 tickets at a 1% rate, and `p_none`, the chance that a random draw of 150 holds none.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Each ticket is other with chance 0.99, independently.

</details>

In [ ]:
# Exercise 7.1: running this cell checks your answer
expected_150 = ...  # your answer
p_none = ...  # your answer



<font size="4"><b>How far one ticket moves recall</b></font>

Store `recall_values_2`, the list of recall values possible with 2 security tickets, and `step_30`, how many percentage points one ticket moves recall with 30.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

With $n$ positives, recall is $k / n$ for $k = 0, \dots, n$.

</details>

In [ ]:
# Exercise 7.2: running this cell checks your answer
recall_values_2 = ...  # your answer
step_30 = ...  # your answer



<font size="4"><b>The binomial</b></font>

Write `binom_pmf(n, p, k)`, the chance of exactly `k` successes in `n` tries, and store `p_20_28`, the chance that a model with true recall 0.8 catches between 20 and 28 of 30, inclusive.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`math.comb(n, k)` counts the ways to choose which `k` tries succeed. `range(20, 29)` stops before 29.

</details>

In [ ]:
# Exercise 7.3: running this cell checks your answer
def binom_pmf(n: int, p: float, k: int) -> float:
    return ...  # your code here


p_20_28 = ...  # your answer



<font size="4"><b>Simulate it</b></font>

Simulate 10,000 test sets of 30 security tickets for a model with true recall 0.8, with `np.random.default_rng(7).binomial`. Store `sim_recall`, the measured recall of each, and `share_in`, the share between 20/30 and 28/30 inclusive. A histogram appears when the check passes.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`.binomial(30, 0.8, 10000)` gives the catches; divide by 30. Compare catches, not recalls, to avoid rounding trouble.

</details>

In [ ]:
# Exercise 7.4: running this cell checks your answer
catches = ...  # your answer
sim_recall = ...  # your answer
share_in = ...  # your answer



<font size="4"><b>A test set of 800</b></font>

A test set holds 800 tickets at a 1% rate. Store `exp_800`, the security tickets you expect, and `step_800`, how many points one of them moves recall.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Start with 1% of 800.

</details>

In [ ]:
# Exercise 7.5: running this cell checks your answer
exp_800 = ...  # your answer
step_800 = ...  # your answer



<font size="4"><b>The spread shrinks</b></font>

For 30 and for 300 security tickets, simulate 10,000 measured recalls at true recall 0.8 with `np.random.default_rng(8)` (a fresh generator for each size). Store `width_30` and `width_300`, the gap between the 97.5th and 2.5th percentiles of each.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The middle 95% runs from the 2.5% quantile to the 97.5% quantile, and `np.quantile` gives each one.

</details>

In [ ]:
# Exercise 7.6: running this cell checks your answer
def spread(n: int) -> float:
    x = ...  # your answer
    return ...  # your code here


width_30, width_300 = ..., ...  # your answer



<font size="4"><b>How many tickets to hold back</b></font>

Write `tickets_needed(k_pos, rate)`, the smallest number of tickets that holds `k_pos` positives on average, and store `need_60`, for 60 positives at 1%.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

You need a whole number of tickets, so round up with `math.ceil`. Round the ratio to a few decimal places first, so that floating point error can't push it up by 1.

</details>

In [ ]:
# Exercise 7.7: running this cell checks your answer
def tickets_needed(k_pos: int, rate: float) -> int:
    return ...  # your code here


need_60 = ...  # your answer



<font size="4"><b>Stratified folds</b></font>

Write `stratified_folds(y, k)`: an array giving each ticket a fold from 0 to `k - 1`, dealing the positives round the folds in turn and the negatives the same way, so every fold gets its share of each. Store `pos_per_fold`, the number of training positives in each of 5 folds.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Within each class, deal the tickets out to the folds in turn. `% k` on a running count does it.

</details>

In [ ]:
# Exercise 7.8: running this cell checks your answer
def stratified_folds(y: np.ndarray, k: int) -> np.ndarray:
    ...  # your code here


pos_per_fold = ...  # your answer



## <font color="#e52e71">Leakage</font>

An earlier version of the model, built by a different engineer, scored far better on validation: all 30 security tickets caught with only 2 false alarms. Precision about 94%, recall 100%.

A score that looks too good is a bug until shown otherwise. The top weights were "security team", "forwarded to" and "hacked". Customers write "hacked". They almost never write "forwarded to the security team".

Draw one ticket's life as a timeline. The customer writes in, and that moment is **prediction time**, when the model must decide. 10 minutes later an agent replies, "forwarded to the security team". 2 days later the case closes. The training data came from an export that glued all the text into one field, so the model read the agent's reply, which already contains the answer.

That is **leakage**: information in the training data that the model won't have when it makes the real decision. Shipped, that model would score well in every report and fail at the desk, because its best feature is always missing there.

The fix is to rebuild every feature from what exists at prediction time: the customer's own text, and facts known at that moment.

Leakage takes many forms. A field filled in later, such as the queue that handled the case. A count that grows over time, such as the number of messages. An identifier that encodes the label, such as a range of ticket numbers used only by the security queue.

Two checks catch most of it. Read the top weights and ask whether each feature could exist at prediction time. And treat any result far better than expected as a bug to explain.

Our data has the agent's reply in `REPLY`, the ticket number in `TICKET_ID` and the message count when the case closed in `N_MSG`. The cell below trains a model on the leaky export.

In [ ]:
EXPORT = np.array([t + " " + r for t, r in zip(TEXT, REPLY)], dtype=object)
VOCAB_LEAK = sorted({w for t in EXPORT[TR] for w in _tokens(t)})
X_LEAK = _counts(EXPORT, VOCAB_LEAK)
W_LEAK, B_LEAK = _train(X_LEAK[TR], Y_TR, w_pos=99.0)
P_LEAK_VA = _prob(X_LEAK[VA], W_LEAK, B_LEAK)
print("one exported ticket:", EXPORT[int(np.flatnonzero(Y == 1)[0])])
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
draw_matrix(axes[0], _cm(Y_VA, _flag(P_LEAK_VA, 0.5)), "Leaky export, validation")
draw_matrix(axes[1], _cm(Y_VA, _flag(P_VA, 0.5)), "Customer text only, validation")
plt.tight_layout()
plt.show()



<font size="4"><b>Read the top weights</b></font>

Write `top_words(w, vocab, k)`: the `k` words with the largest weights, largest first. Store `leak_top5`, the top 5 of the leaky model `W_LEAK` over `VOCAB_LEAK`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`np.argsort` puts the smallest first. Sort the negated weights to get the largest first, and keep the first `k`.

</details>

In [ ]:
# Exercise 8.1: running this cell checks your answer
def top_words(w: np.ndarray, vocab: list[str], k: int) -> list[str]:
    return ...  # your code here


leak_top5 = ...  # your answer



<font size="4"><b>The video&#x27;s leaky score</b></font>

The leaky model's validation matrix in the video was (30, 0, 2, 2968). Store `leak_p` and `leak_r`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Your `precision` and `recall`.

</details>

In [ ]:
# Exercise 8.2: running this cell checks your answer
leak_p, leak_r = ..., ...  # your answer



<font size="4"><b>Our leaky model, priced</b></font>

Store `leak_cost` and `clean_cost`, the validation costs at a threshold of 0.5 of the leaky model (`P_LEAK_VA`) and the clean one (`P_VA`).

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Your `cost_at` twice.

</details>

In [ ]:
# Exercise 8.3: running this cell checks your answer
leak_cost = ...  # your answer
clean_cost = ...  # your answer



<font size="4"><b>Which fields exist at prediction time</b></font>

Store `at_prediction`, a dictionary from each of these fields to `True` if it is known when the ticket arrives: `"customer's text"`, `"account age"`, `"agent's reply"`, `"messages when the case closed"`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Place each one on the timeline: arrival, reply 10 minutes later, close 2 days later.

</details>

In [ ]:
# Exercise 8.4: running this cell checks your answer
at_prediction = ...  # your answer



<font size="4"><b>Filter by the timeline</b></font>

Write `known_at(field_times, t_pred)`: the sorted list of fields whose time (minutes after arrival) is at or before `t_pred`. Store `usable`, the result on `FIELD_TIMES` at prediction time 0.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

A list comprehension over `field_times.items()`, then `sorted`.

</details>

In [ ]:
# Exercise 8.5: running this cell checks your answer
FIELD_TIMES = {"customer text": 0, "customer country": 0, "agent reply": 10, "refund issued": 300,
               "queue that handled it": 2880, "messages at close": 2880}


def known_at(field_times: dict[str, int], t_pred: int) -> list[str]:
    return ...  # your code here


usable = ...  # your answer



<font size="4"><b>A ticket number that leaks</b></font>

Ticket numbers from the security queue start at 900,000. Store `id_pr`, the (precision, recall) on validation of the rule "flag when `TICKET_ID` is at least 900,000".

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`np.where(TICKET_ID[VA] >= 900000, 1, -1)`.

</details>

In [ ]:
# Exercise 8.6: running this cell checks your answer
cm_id = ...  # your answer
id_pr = ...  # your answer



<font size="4"><b>A count that grows</b></font>

Store `msg_means`, the mean of `N_MSG` over (security, other) tickets in the training split, and `msg_pr`, the validation (precision, recall) of the rule "flag when `N_MSG` is at least 6".

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Mask with `Y_TR == 1` for the first mean. Build the rule's predictions like the last exercise.

</details>

In [ ]:
# Exercise 8.7: running this cell checks your answer
msg_means = ...  # your answer
cm_msg = ...  # your answer
msg_pr = ...  # your answer



<font size="4"><b>Words only the agent writes</b></font>

Store `reply_only`, the sorted list of words that appear in some `REPLY` and in no customer `TEXT`. Then store `leaky_in_top`, how many of the leaky model's top 10 words are in that list.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Build two sets of words with set comprehensions, take the difference, then `sorted`. Your `top_words` gives the top 10.

</details>

In [ ]:
# Exercise 8.8: running this cell checks your answer
reply_only = ...  # your answer
leaky_in_top = ...  # your answer



## <font color="#00897b">Leaks in the split</font>

Customers who get no answer often write again. One customer sent the same takeover ticket 3 times in an hour, with small changes. Tickets that match each other apart from a few words are **near-duplicates**.

Split at random and copies fall on both sides: two in training, one in test. The model has already seen the answer for that exact ticket, and scores it easily. In the video's random split, 12 of the 30 test positives had a copy in training and all 12 were caught; of the 18 with no copy, 15 were caught. The team would report 27 of 30, 90%. On genuinely new tickets, recall is 15 of 18, about 83%.

Random splits leak in time too. Tickets from December land in training and tickets from March in test, so the model learns from the future, including scams that hadn't started yet.

The fix is a **time-based split**: sort the tickets by date and cut the calendar. Train on the earliest months, validate on the next stretch, test on the most recent. That copies how the model will be used: trained on the past, run on the future. And keep each customer's tickets on one side of the cut, so no near-duplicate crosses it.

To find near-duplicates in code, compare word sets. The **Jaccard similarity** of two tickets is the number of distinct words they share divided by the number of distinct words in either:
$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}.$$
It is 1 for the same set of words and 0 for no words in common. We call two tickets near-duplicates when $J \ge 0.7$.

In [ ]:
dup = int(np.flatnonzero((CUSTOMER[:-1] == CUSTOMER[1:]) & (Y[:-1] == 1) & (Y[1:] == 1))[0])
print("a customer writing twice on day", DAY[dup])
print("  ", TEXT[dup]); print("  ", TEXT[dup + 1])
fig, ax = plt.subplots(figsize=(9, 2))
ax.scatter(DAY, np.zeros(N_ALL) + np.random.default_rng(1).uniform(-1, 1, N_ALL) * 0.3, s=1, color=GREY)
for lo, hi, colour, name in [(0, 14000, PALETTE["cyan"], "train"), (14000, 17000, PALETTE["amber"], "validation"), (17000, 20000, PALETTE["pink"], "test")]:
    ax.axvspan(DAY[lo], DAY[hi - 1], color=colour, alpha=0.25)
    ax.text((DAY[lo] + DAY[hi - 1]) / 2, 0.45, name, ha="center", fontsize=9)
ax.set_yticks([]); ax.set_xlabel("day of the year"); ax.set_title("The split by date")
plt.show()



<font size="4"><b>Jaccard similarity</b></font>

Write `jaccard(a, b)` for two ticket texts, using your `tokenize`. Return 1.0 if both are empty.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Python sets have `&` for the shared items and `|` for all items. Handle two empty tickets first.

</details>

In [ ]:
# Exercise 9.1: running this cell checks your answer
def jaccard(a: str, b: str) -> float:
    ...  # your code here



<font size="4"><b>Has a copy</b></font>

Write `has_copy(a_texts, b_texts, cut=0.7)`: a boolean array, one entry per text in `a_texts`, True when some text in `b_texts` has Jaccard similarity at least `cut` with it.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Loop over `a_texts`. `any(...)` over a generator of comparisons with your `jaccard` stops at the first match.

</details>

In [ ]:
# Exercise 9.2: running this cell checks your answer
def has_copy(a_texts, b_texts, cut: float = 0.7) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>A random split</b></font>

Split the tickets at random with `perm = np.random.default_rng(5).permutation(N_ALL)`: training is `perm[:14000]`, test is `perm[17000:]`. Store `rand_copy_share`, the share of test security tickets that have a copy among the training security tickets.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Compare positives with positives only; that keeps it fast. `TEXT[idx][Y[idx] == 1]` picks the security texts of a split.

</details>

In [ ]:
# Exercise 9.3: running this cell checks your answer
perm = np.random.default_rng(5).permutation(N_ALL)
tr_r, te_r = ..., ...  # your answer
rand_copy_share = ...  # your answer



<font size="4"><b>The date split</b></font>

Store `date_copy_share`, the same share for the date split: test is `TE`, training is `TR`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The same line as before, with the slices.

</details>

In [ ]:
# Exercise 9.4: running this cell checks your answer
date_copy_share = ...  # your answer



<font size="4"><b>Reported and honest recall</b></font>

In the video's random split, 12 test positives had a copy in training (all 12 caught) and 15 of the other 18 were caught. Store `reported`, the recall over all 30, and `honest`, the recall on the 18 with no copy.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Add the catches first, then look only at the 18.

</details>

In [ ]:
# Exercise 9.5: running this cell checks your answer
reported = ...  # your answer
honest = ...  # your answer



<font size="4"><b>Another split, the same question</b></font>

6 test positives have a copy in training and all 6 are caught. Of the other 24, 18 are caught. Store `reported_b` and `honest_b`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Add first, then look only at the 24.

</details>

In [ ]:
# Exercise 9.6: running this cell checks your answer
reported_b = ...  # your answer
honest_b = ...  # your answer



<font size="4"><b>Learning from the future</b></font>

Store `future_share`, the share of test tickets in the random split (`te_r`) that are dated earlier than the latest training ticket in `tr_r`, and `future_share_date`, the same for `TE` and `TR`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

Compare each test day with the latest training day, then take the mean of the booleans.

</details>

In [ ]:
# Exercise 9.7: running this cell checks your answer
future_share = ...  # your answer
future_share_date = ...  # your answer



<font size="4"><b>Group by customer</b></font>

Make a random split that keeps each customer on one side: a ticket goes to test when `CUSTOMER % 5 == 0`, otherwise to training. Store `group_copy_share`, the share of test security tickets with a copy among training security tickets.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`np.flatnonzero` on the condition and on its opposite gives the two sides. Then `has_copy` on the positives as before.

</details>

In [ ]:
# Exercise 9.8: running this cell checks your answer
te_g, tr_g = ..., ...  # your answer
group_copy_share = ...  # your answer



## <font color="#ff4d6d">Error analysis</font>

**Error analysis** means reading a model's mistakes, sorting them into groups and counting each group. The counts say how often the model is wrong, and the tickets themselves say why.

The main model's validation mistakes were 6 missed security tickets and 120 false alarms. Reading 120 tickets takes a while, so draw a random 50 of the false alarms. Sorted as you read: 28 are password resets the customer asked for, 12 are cards declined while travelling, 6 ask whether an email they received is real, 4 are other things. Of the 6 misses: 3 are short tickets from the phone app, 2 describe purchases in wording the model never saw, and 1 is no takeover at all, a wrong label.

The sample stands for all 120. Password resets are 28 of 50, so about 67 of the 120 false alarms are password resets.

Before building a fix, compute a **ceiling**: the most the fix could gain if it worked perfectly. Remove all 67 estimated password-reset false alarms, and precision rises from about 17% to about 31%. On the miss side, catching all 3 short app tickets would raise recall from 80% to 90%.

Ceilings let you compare fixes before spending a week on any of them. A group of 2 tickets has a small ceiling however interesting its mistakes are.

All of this reads validation mistakes. Reading the test set's mistakes and fixing them would turn the test set into one more validation set.

Our synthetic data records each ticket's topic in `TOPIC`. The model never sees it; here it does the job of a person reading the ticket and naming its group.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].barh(["password reset", "card declined", "is this real?", "other"][::-1], [28, 12, 6, 4][::-1], color=PALETTE["amber"])
axes[0].set_title("50 sampled false alarms")
axes[1].barh(["short app ticket", "new wording", "wrong label"][::-1], [3, 2, 1][::-1], color=RED)
axes[1].set_title("The 6 misses")
plt.tight_layout()
plt.show()



<font size="4"><b>Scale the sample up</b></font>

Store `pw_est`, the estimated number of password-reset false alarms among all 120, rounded to a whole ticket.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

28 of 50, applied to 120.

</details>

In [ ]:
# Exercise 10.1: running this cell checks your answer
pw_est = ...  # your answer



<font size="4"><b>The precision ceiling</b></font>

Write `ceiling_precision(tp, fp, removed)`: the precision if `removed` false alarms disappeared. Store `pw_ceiling` for the main model with your `pw_est`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The flagged count drops by `removed`; the true positives stay.

</details>

In [ ]:
# Exercise 10.2: running this cell checks your answer
def ceiling_precision(tp: int, fp: int, removed: float) -> float:
    return ...  # your code here


pw_ceiling = ...  # your answer



<font size="4"><b>The recall ceiling</b></font>

Store `rec_ceiling`, the main model's recall if all 3 short app tickets were caught as well.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

3 more true positives, the same 30 security tickets.

</details>

In [ ]:
# Exercise 10.3: running this cell checks your answer
rec_ceiling = ...  # your answer



<font size="4"><b>The travel group</b></font>

Store `travel_est`, the estimated number of travel false alarms among all 120 (12 in the sample of 50), rounded, and `travel_vs_pw`, its size relative to the password-reset estimate.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Scale the sample to the whole, then divide by `pw_est`.

</details>

In [ ]:
# Exercise 10.4: running this cell checks your answer
travel_est = ...  # your answer
travel_vs_pw = ...  # your answer



<font size="4"><b>Sample our false alarms</b></font>

Take our model's validation false alarms at a threshold of 0.5. Draw 50 of them with `np.random.default_rng(10).choice(fa_idx, 50, replace=False)`, where `fa_idx` holds their positions within the validation split. Store `fa_counts`, a dictionary from topic to count in the sample.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Mask the other tickets at or above 0.5 and take `np.flatnonzero`. `np.unique(..., return_counts=True)` returns names and counts; `zip` them into a dictionary.

</details>

In [ ]:
# Exercise 10.5: running this cell checks your answer
fa_idx = ...  # your answer
sample = ...  # your answer
fa_counts = ...  # your answer



<font size="4"><b>Our biggest group and its ceiling</b></font>

Store `big_topic`, the most common topic in your sample, `big_est`, its estimated count among all our validation false alarms (not rounded), and `big_ceiling`, our model's precision at 0.5 if all of them went away.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`max` with a `key` finds the biggest group. Scale its count from the sample of 50 up to all the false alarms, then use your `ceiling_precision`.

</details>

In [ ]:
# Exercise 10.6: running this cell checks your answer
big_topic = ...  # your answer
big_est = ...  # your answer
big_ceiling = ...  # your answer



<font size="4"><b>Rank fixes by their ceiling</b></font>

Write `rank_fixes(groups, tp, fp)`: given a dictionary from group name to its estimated false alarms, return the group names sorted by precision ceiling, highest first. Store `video_rank` for the video's groups scaled to 120: `VIDEO_FA`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`sorted` takes a `key` function and `reverse=True`. Use your `ceiling_precision` inside the key.

</details>

In [ ]:
# Exercise 10.7: running this cell checks your answer
VIDEO_FA = {"password reset": 28 / 50 * 120, "card declined": 12 / 50 * 120, "is this real?": 6 / 50 * 120, "other": 4 / 50 * 120}


def rank_fixes(groups: dict[str, float], tp: int, fp: int) -> list[str]:
    return ...  # your code here


video_rank = ...  # your answer



<font size="4"><b>Our misses</b></font>

Store `miss_topics`, a dictionary from topic to count over our model's missed validation security tickets at a threshold of 0.5, and `n_missed`, how many there are.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The misses are `(Y_VA == 1) & (P_VA < 0.5)`.

</details>

In [ ]:
# Exercise 10.8: running this cell checks your answer
missed = ...  # your answer
n_missed = ...  # your answer
miss_topics = ...  # your answer



## <font color="#1e88e5">Label agreement</font>

Every label in the data was set by a person reading a ticket. People get tired, read quickly and disagree about borderline cases.

To measure that, take 400 tickets and have 2 people label each one without seeing each other's answers. The first reader flags 5 as security, the second flags 6, and 4 are flagged by both.

**Label agreement** is the share of tickets on which the 2 readers give the same label. Overall they agree on 397 of 400, over 99%. That number sounds reassuring, and it hides the problem the same way accuracy did.

Look only at the tickets someone flagged. There are 7, and the readers agree on 4: about 57%. If 2 careful people agree on only 4 of 7 security tickets, no model can be scored more finely than that. Some of the model's misses and false alarms are disagreements about the label.

The fix is a short **labelling guide** with worked examples of the hard cases. A customer who asked for a reset and then can't log in is other. A customer who reports a reset they never asked for is security. Relabel the disputed tickets with the guide, measure agreement again, and check it whenever new people join the labelling.

A common summary that corrects for chance is **Cohen's kappa**. If $p_o$ is the observed agreement and $p_e$ the agreement two readers would reach by chance, labelling at their own rates independently, then
$$\kappa = \frac{p_o - p_e}{1 - p_e}.$$
It is 1 for perfect agreement and 0 for agreement no better than chance.

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 3))
ax.bar(["overall\n397 of 400", "flagged by anyone\n4 of 7"], [397 / 400, 4 / 7], color=[PALETTE["mint"], PALETTE["pink"]])
ax.set_ylim(0, 1.05); ax.set_ylabel("agreement")
plt.show()



<font size="4"><b>Overall agreement</b></font>

From 400 tickets, reader A flags 5, reader B flags 6, and 4 are flagged by both. Store `n_disagree`, the tickets where they differ, and `agree_all`, the overall agreement.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

A ticket is disputed when exactly one reader flagged it.

</details>

In [ ]:
# Exercise 11.1: running this cell checks your answer
n_disagree = ...  # your answer
agree_all = ...  # your answer



<font size="4"><b>Agreement on the flagged tickets</b></font>

Store `n_anyone`, the tickets flagged by at least one reader, and `agree_flagged`, the agreement among them.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Flagged by A plus flagged by B, minus those counted twice.

</details>

In [ ]:
# Exercise 11.2: running this cell checks your answer
n_anyone = ...  # your answer
agree_flagged = ...  # your answer



<font size="4"><b>Another pair of readers</b></font>

500 tickets are labelled twice. Reader A flags 8, reader B flags 6, and 5 are flagged by both. Store `agree_flagged_b` and `agree_all_b`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Count the tickets anyone flagged first.

</details>

In [ ]:
# Exercise 11.3: running this cell checks your answer
agree_flagged_b = ...  # your answer
agree_all_b = ...  # your answer



<font size="4"><b>Agreement in code</b></font>

Write `agreement(a, b)` for two label arrays of $+1$ and $-1$, returning `(overall, on_flagged)`, where `on_flagged` uses only tickets that at least one reader flagged.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Build a mask of tickets that either reader flagged, and compare the two arrays only there.

</details>

In [ ]:
# Exercise 11.4: running this cell checks your answer
def agreement(a, b) -> tuple[float, float]:
    ...  # your code here



<font size="4"><b>Two readers on our validation set</b></font>

`READER_A` is the validation labels. `READER_B` is a second reader who disagrees on some borderline tickets. Store `agree_va`, your `agreement` of the two.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

One call.

</details>

In [ ]:
# Exercise 11.5: running this cell checks your answer
r_lab = np.random.default_rng(12)
READER_A = Y_VA.copy()
READER_B = Y_VA.copy()
border = np.flatnonzero(np.isin(TOPIC[VA], ["phish", "reset"]))
READER_B[r_lab.choice(border, 4, replace=False)] = 1
READER_B[r_lab.choice(np.flatnonzero((Y_VA == 1) & (TOPIC[VA] == "short")), 2, replace=False)] = -1
agree_va = ...  # your answer



<font size="4"><b>Cohen&#x27;s kappa</b></font>

Write `kappa(a, b)`. The chance agreement is $p_e = q_a q_b + (1 - q_a)(1 - q_b)$, where $q_a$ and $q_b$ are each reader's share of $+1$. Store `kappa_video` for the 400 tickets `A` and `B` from the check above, rebuilt here.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`pa, pb = np.mean(a == 1), np.mean(b == 1)`; `po = np.mean(a == b)`.

</details>

In [ ]:
# Exercise 11.6: running this cell checks your answer
A400 = -np.ones(400, dtype=int); B400 = -np.ones(400, dtype=int)
A400[[0, 1, 2, 3, 4]] = 1; B400[[0, 1, 2, 3, 5, 6]] = 1


def kappa(a, b) -> float:
    ...  # your code here


kappa_video = ...  # your answer



<font size="4"><b>Wrong labels and recall</b></font>

Of 30 test positives, 3 have wrong labels: they are other tickets, and the model missed all 3. It caught 24. Store `rec_measured`, recall against the labels as given, and `rec_corrected`, recall against the corrected labels.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Correcting the labels removes 3 tickets from the top row, all of them misses.

</details>

In [ ]:
# Exercise 11.7: running this cell checks your answer
rec_measured = ...  # your answer
rec_corrected = ...  # your answer



<font size="4"><b>The disputed tickets</b></font>

Store `disputed`, the positions within validation where `READER_A` and `READER_B` disagree, and `disputed_topics`, a dictionary from topic to count over them.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`np.flatnonzero(READER_A != READER_B)`, then `np.unique` on their topics.

</details>

In [ ]:
# Exercise 11.8: running this cell checks your answer
disputed = ...  # your answer
disputed_topics = ...  # your answer



## <font color="#d84315">Drift</font>

**Drift** is a change in the data after launch. The model stays fixed and the tickets change under it. There are 2 kinds, and they do different damage.

Start from what the model does on each class. On security tickets it catches 80%: its recall. On other tickets it flags 120 of 2,970, about 4%. That is the **false alarm rate**: the share of negatives it flags. Both rates belong to the model, and they hold as long as the tickets in each class look the way they did in training.

The first kind of drift is a change in the mix. A scam wave starts and the base rate rises from 1% to 3%. On a week of 3,000 tickets, 90 are security and the model catches 72. About 118 of the others are flagged. Precision rises from about 17% to about 38%. The model's rates are unchanged; precision moved because there are more takeovers to find. In a quiet week precision falls, and the team may think the model is failing when nothing about it changed.

Precision depends on 3 things: recall, the false alarm rate and the base rate. With $n$ tickets at base rate $\pi$,
$$\text{precision} = \frac{\pi \cdot \text{recall}}{\pi \cdot \text{recall} + (1 - \pi) \cdot \text{FAR}}.$$
A precision measured in testing holds only at the base rate it was measured at.

The second kind is a change in meaning. The scam wave brings a new trick: the attacker asks the customer to read back a verification code sent by text message. The tickets talk about codes and texts, words that were rare in training. On 10 such tickets the model catches 2: recall 20% on the new kind.

So the 2 kinds show up differently. A change in mix moves precision and leaves recall alone. A change in meaning lowers recall on the new cases, and nothing in the flagged tickets announces it, because the misses never reach the security team. Missed takeovers are silent.

In [ ]:
rates = np.linspace(0.002, 0.06, 100)
rec0, far0 = 0.8, 120 / 2970
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(rates * 100, rates * rec0 / (rates * rec0 + (1 - rates) * far0), color=PALETTE["violet"])
for r_, name in [(0.005, "0.5%"), (0.01, "1%"), (0.03, "3%")]:
    p_ = r_ * rec0 / (r_ * rec0 + (1 - r_) * far0)
    ax.scatter([r_ * 100], [p_], color=PALETTE["amber"], zorder=4)
    ax.annotate(f"{name}: {p_:.0%}", (r_ * 100, p_), xytext=(6, -12), textcoords="offset points", fontsize=8)
ax.set_xlabel("base rate (%)"); ax.set_ylabel("precision"); ax.set_title("Same model, different weeks")
plt.show()



<font size="4"><b>The false alarm rate</b></font>

Store `far`, the main model's false alarm rate from `MAIN`, and `rec`, its recall.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

False alarms over all other tickets.

</details>

In [ ]:
# Exercise 12.1: running this cell checks your answer
far = ...  # your answer
rec = ...  # your answer



<font size="4"><b>A matrix at any base rate</b></font>

Write `at_rate(rate, n, rec, far)` returning the expected `(tp, fn, fp, tn)` as floats for `n` tickets at base rate `rate`.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

Positives are `n * rate`. Of them, `rec` are caught. Of the negatives, `far` are flagged.

</details>

In [ ]:
# Exercise 12.2: running this cell checks your answer
def at_rate(rate: float, n: int, rec: float, far: float):
    ...  # your code here



<font size="4"><b>A scam wave</b></font>

Store `cm_3pct`, the expected matrix on 3,000 tickets at a 3% base rate with the main model's rates, and `prec_3pct`, its precision.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

Your `at_rate`, then `precision(tp, fp)`.

</details>

In [ ]:
# Exercise 12.3: running this cell checks your answer
cm_3pct = ...  # your answer
prec_3pct = ...  # your answer



<font size="4"><b>A quiet week</b></font>

Store `prec_half`, the precision at a base rate of 0.5% on 3,000 tickets with the same rates.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

Build the matrix first.

</details>

In [ ]:
# Exercise 12.4: running this cell checks your answer
cm_half = ...  # your answer
prec_half = ...  # your answer



<font size="4"><b>Precision over base rates</b></font>

Store `prec_curve`, the precision at each base rate in `RATES` with the main model's rates, as a NumPy array. A plot appears when the check passes.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

The formula in the text, with arrays.

</details>

In [ ]:
# Exercise 12.5: running this cell checks your answer
RATES = np.linspace(0.001, 0.1, 100)
prec_curve = ...  # your answer



<font size="4"><b>The new wording</b></font>

On 10 tickets with the new scam's wording, the model catches 2. Store `rec_new_video`.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

Caught over all.

</details>

In [ ]:
# Exercise 12.6: running this cell checks your answer
rec_new_video = ...  # your answer



<font size="4"><b>A live week with a new scam</b></font>

`LIVE_TEXT`, `LIVE_Y` and `LIVE_TOPIC` are one synthetic week after launch: a 3% base rate, and 40% of the security tickets use the new wording (topic `"newscam"`). Store `rec_old` and `rec_new`, our model's recall at `t_star` on the older kinds of security ticket and on the new one.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

`P_LIVE = _prob(_counts(LIVE_TEXT, VOCAB), W_MAIN, B_MAIN)`. The new ones are `LIVE_TOPIC == "newscam"`; the older ones are the other positives.

</details>

In [ ]:
# Exercise 12.7: running this cell checks your answer
LIVE_TEXT, LIVE_Y, LIVE_TOPIC = make_week(91, 3000, 0.03, 0.4)
P_LIVE = ...  # your answer
new_kind = ...  # your answer
rec_old = ...  # your answer
rec_new = ...  # your answer



<font size="4"><b>Words the model has never seen</b></font>

Write `unseen_share(texts, vocab)`: the share of all words in the texts that are not in `vocab`. Store `unseen_val` for the validation texts and `unseen_live` for the live week. This needs no labels.

<details><summary><font color="#d84315"><b>💡 Hint</b></font></summary>

Make a set from `vocab`, then count over every word of every ticket from your `tokenize`.

</details>

In [ ]:
# Exercise 12.8: running this cell checks your answer
def unseen_share(texts, vocab) -> float:
    ...  # your code here


unseen_val = ...  # your answer
unseen_live = ...  # your answer



## <font color="#2e7d32">Monitoring</font>

**Monitoring** means measuring the model on live data, every week, after launch. Some measurements are cheap, and some cost hours of labelling.

The cheapest is the share of tickets flagged. It needs no labels: each week, divide the flagged tickets by the total. For 8 weeks it sits close to 4.8%. In week 9 it jumps to 7.5%: something changed, and the jump says which week to examine.

The second line is precision, from the security team's own verdicts on the tickets they receive. They decide on every flagged ticket anyway, so it costs nothing extra. It sits near 17%, then reaches 36% in week 9: more takeovers in the mix.

Recall is the expensive one, because misses never reach the team. To estimate it, hand-label a random sample of the tickets the model didn't flag: here, 500 unflagged tickets in weeks 4, 8 and 10. Scale the misses found up to all unflagged tickets, and
$$\text{recall} \approx \frac{\text{caught}}{\text{caught} + \text{estimated misses}}.$$
In weeks 4 and 8 each sample finds 1 missed takeover and recall is about 80%. In week 10 the sample finds 2, and estimated recall drops to about 69%. Week 10's flagged share is back to 4.8%: the cheap line looks normal while recall falls.

**Retraining** is the response: fit the model again on fresh labelled tickets, including the new scam, with the same split by date and the same checks. Many teams retrain on a schedule and also whenever a monitoring line crosses an alarm level. Set those alarm levels before launch, from the validation results: for example, a flagged share above 6% or a sampled recall below 70%.

`WEEKS` holds (tickets, flagged) for 10 weeks and `CONFIRMED` the security team's confirmed takeovers among the flagged.

In [ ]:
WEEKS = [(3000, 144), (2950, 140), (3050, 149), (3000, 141), (3100, 150), (2980, 142), (3020, 146), (3000, 145), (3000, 225), (3040, 146)]
CONFIRMED = [24, 23, 25, 23, 25, 24, 24, 25, 81, 26]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(range(1, 11), [100 * f / n for n, f in WEEKS], marker="o", color=PALETTE["cyan"])
axes[0].axhline(6, color=PALETTE["pink"], ls="--", lw=1)
axes[0].set_title("Flagged share (%), alarm at 6%"); axes[0].set_xlabel("week")
axes[1].plot(range(1, 11), [100 * c / f for (n, f), c in zip(WEEKS, CONFIRMED)], marker="o", color=PALETTE["amber"])
axes[1].set_title("Precision from the team's verdicts (%)"); axes[1].set_xlabel("week")
plt.tight_layout()
plt.show()



<font size="4"><b>The flagged share</b></font>

Store `week_share`, the flagged share of each week in percent, as a NumPy array.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

A percentage for each `(tickets, flagged)` pair in `WEEKS`, collected into an array.

</details>

In [ ]:
# Exercise 13.1: running this cell checks your answer
week_share = ...  # your answer



<font size="4"><b>Which weeks raise the alarm</b></font>

Store `alarm_weeks`, the list of week numbers (starting at 1) whose flagged share is above 6%.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

`np.flatnonzero` gives positions, and positions start at 0.

</details>

In [ ]:
# Exercise 13.2: running this cell checks your answer
alarm_weeks = ...  # your answer



<font size="4"><b>Precision from the team&#x27;s verdicts</b></font>

Store `week_prec`, each week's precision in percent.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

Confirmed over flagged.

</details>

In [ ]:
# Exercise 13.3: running this cell checks your answer
week_prec = ...  # your answer



<font size="4"><b>Recall from a sample</b></font>

Write `sampled_recall(n, flagged, caught, found, sample)`: scale the `found` misses in a sample of `sample` unflagged tickets up to all `n - flagged` unflagged tickets, and return the estimated recall. Store `rec_w10` for week 10, where the sample of 500 found 2.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

Estimated misses are `found * (n - flagged) / sample`. Week 10 is `WEEKS[9]` with `CONFIRMED[9]` caught.

</details>

In [ ]:
# Exercise 13.4: running this cell checks your answer
def sampled_recall(n: int, flagged: int, caught: int, found: int, sample: int) -> float:
    missed = ...  # your answer
    return ...  # your code here


rec_w10 = ...  # your answer



<font size="4"><b>One week&#x27;s share</b></font>

One week brings 3,200 tickets and 240 are flagged. Store `share_one` in percent.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

Flagged over total.

</details>

In [ ]:
# Exercise 13.5: running this cell checks your answer
share_one = ...  # your answer



<font size="4"><b>How unusual is week 9</b></font>

Using weeks 1 to 8 as the normal range, store `z9`, how many standard deviations week 9's flagged share sits above their mean, and `z10` for week 10. Use `np.std` with its default.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

Weeks 1 to 8 are `week_share[:8]`. Subtract their mean and divide by their standard deviation.

</details>

In [ ]:
# Exercise 13.6: running this cell checks your answer
z9 = ...  # your answer
z10 = ...  # your answer



<font size="4"><b>The quiet failure</b></font>

Store `quiet_week`, the week whose flagged share is under the 6% alarm while its sampled recall is under the 70% alarm. Sampled recall was measured in weeks 4, 8 and 10, finding 1, 1 and 2 misses in 500.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

Compute the sampled recall for the three weeks and compare both lines with their alarm levels.

</details>

In [ ]:
# Exercise 13.7: running this cell checks your answer
FOUND = {4: 1, 8: 1, 10: 2}
quiet_week = ...  # your answer



<font size="4"><b>How many tickets to label</b></font>

At week 10's miss rate in the unflagged tickets (2 in 500), store `label_for_5`, how many unflagged tickets you'd label to expect 5 misses in the sample.

<details><summary><font color="#2e7d32"><b>💡 Hint</b></font></summary>

The sample found 2 misses in 500 tickets. How many tickets give 5 misses at that rate?

</details>

In [ ]:
# Exercise 13.8: running this cell checks your answer
label_for_5 = ...  # your answer



## <font color="#5e35b1">The pipeline and the payoff</font>

Here is the whole pipeline, with the number each box produced in this case:

1. The decision and its costs: flag for the security team, a miss charged 50, a needless check charged 1.
2. Labelled data, with an agreement check on the rare class: 4 of 7 flagged tickets agreed, and a labelling guide followed.
3. The split by date, each customer's tickets kept together: 14,000, 3,000 and 3,000.
4. Baselines. The always-other model sets the floor. A keyword rule, flag any ticket with "hacked" or "stolen", is a second baseline that takes 5 minutes to build. A trained model has to beat both on cost.
5. Features built only from what exists when the ticket arrives: the customer's words.
6. Training: the logistic loss with a class weight of 99 on security tickets.
7. The threshold, chosen on validation by total cost.
8. Error analysis on the validation mistakes, with a ceiling for each fix.
9. One look at the test set: in the video, 24 of 30 caught, 120 false alarms, a cost of 420.
10. Launch, with weekly monitoring and a retraining schedule.

Monitoring feeds new labels back to box 2, and the pipeline runs again. Every box before the test set may be repeated as often as you like. The test set is opened once, after everything before it is fixed.

Back to the opening. The first model: 99% accurate, 0 of 30 caught, a cost of 1,500. The new model: 95.8% accurate, 24 of 30 caught, a cost of 420. On accuracy the first model wins. On recall, precision and cost, the numbers the help desk wrote down before training, the new model wins by a wide margin. And its score holds up because it was earned honestly: features from arrival time, a split by date, no near-duplicates across it, a threshold chosen on validation and a test set opened once.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar(["always other\n99% accurate", "main model\n95.8% accurate"], [1500, 420], color=[GREY, PALETTE["mint"]])
for k, v in enumerate([1500, 420]):
    ax.text(k, v + 30, f"cost {v:,}", ha="center")
ax.set_ylabel("total cost on 3,000 test tickets"); ax.set_ylim(0, 1700)
plt.show()



<font size="4"><b>Put the boxes in order</b></font>

`BOXES` lists the pipeline's boxes out of order. Store `pipeline_order`, the same strings in the order of the pipeline.

<details><summary><font color="#5e35b1"><b>💡 Hint</b></font></summary>

Decision first, launch last. Baselines come before features and training; the test set comes after error analysis.

</details>

In [ ]:
# Exercise 14.1: running this cell checks your answer
BOXES = ["threshold by cost", "test set once", "split by date", "decision and costs", "training with a class weight",
         "launch and monitor", "baselines", "error analysis", "labels and agreement", "features at arrival"]
pipeline_order = ...  # your answer



<font size="4"><b>The keyword baseline</b></font>

Store `kw_cm`, the test confusion matrix of the rule "flag a ticket whose words include `hacked` or `stolen`", and `kw_cost`, its cost.

<details><summary><font color="#5e35b1"><b>💡 Hint</b></font></summary>

The intersection of `{"hacked", "stolen"}` with a ticket's set of words is empty when neither word appears.

</details>

In [ ]:
# Exercise 14.2: running this cell checks your answer
kw_pred = ...  # your answer
kw_cm = ...  # your answer
kw_cost = ...  # your answer



<font size="4"><b>Too good to be true</b></font>

A team skips the date split and the leakage check, and reports 30 of 30 caught on validation. Store `send_back`, the set of two box names from `BOXES` you'd send them back to.

<details><summary><font color="#5e35b1"><b>💡 Hint</b></font></summary>

A perfect score on a rare class points first to near-duplicates across the split, and to features from after prediction time.

</details>

In [ ]:
# Exercise 14.3: running this cell checks your answer
send_back = ...  # your answer



<font size="4"><b>Open the test set once</b></font>

Store `final_cm`, our model's test confusion matrix at `t_star`, and `final_cost`, its cost.

<details><summary><font color="#5e35b1"><b>💡 Hint</b></font></summary>

Your `confusion` and `flag`, with `t_star` from validation.

</details>

In [ ]:
# Exercise 14.4: running this cell checks your answer
final_cm = ...  # your answer
final_cost = ...  # your answer



<font size="4"><b>Beat both baselines</b></font>

Store `test_costs`, a dictionary with the test costs of `"always other"`, `"keyword"` and `"model"`, and `winner`, the key with the lowest cost.

<details><summary><font color="#5e35b1"><b>💡 Hint</b></font></summary>

The always-other model misses every test positive. `min(test_costs, key=test_costs.get)`.

</details>

In [ ]:
# Exercise 14.5: running this cell checks your answer
test_costs = ...  # your answer
winner = ...  # your answer



<font size="4"><b>The payoff in the video&#x27;s numbers</b></font>

Store `acc_pair`, the accuracies of `BASE` and `MAIN`; `cost_pair`, their costs; and `saved`, how much less the main model costs.

<details><summary><font color="#5e35b1"><b>💡 Hint</b></font></summary>

Your `accuracy` and `total_cost`.

</details>

In [ ]:
# Exercise 14.6: running this cell checks your answer
acc_pair = ...  # your answer
cost_pair = ...  # your answer
saved = ...  # your answer



<font size="4"><b>Over a year</b></font>

The help desk sees 20,000 tickets a year. Store `saved_year`, the saving from the last exercise scaled from 3,000 tickets to 20,000.

<details><summary><font color="#5e35b1"><b>💡 Hint</b></font></summary>

The saving is for 3000 tickets, and a year has 20000.

</details>

In [ ]:
# Exercise 14.7: running this cell checks your answer
saved_year = ...  # your answer



<font size="4"><b>An honest report</b></font>

Write `report(y, probs, t)` returning a dictionary with keys `"matrix"`, `"accuracy"`, `"precision"`, `"recall"` and `"cost"`, for flagging at `t`. Store `final_report` for our model on the test set at `t_star`.

<details><summary><font color="#5e35b1"><b>💡 Hint</b></font></summary>

Everything you need is already a function: `confusion`, `flag`, `accuracy`, `precision`, `recall`, `total_cost`.

</details>

In [ ]:
# Exercise 14.8: running this cell checks your answer
def report(y, probs, t: float) -> dict:
    ...  # your code here


final_report = ...  # your answer



## <font color="#ad1457">What you built</font>

* A decision written down first: two labels, the positive class, the prices of each mistake and the base rate.
* The confusion matrix, and why accuracy flatters a model that ignores a 1% class.
* Precision and recall, which read one column and one row and never see the easy true negatives.
* Thresholds and the precision-recall curve: one trained model, a whole curve of classifiers.
* A threshold chosen on validation by total cost, with F1 as the fallback when nobody can name the costs.
* Class weights in PyTorch, resampling, and a test set kept at the natural rate.
* How few positives a test set holds, and how much one ticket moves recall.
* Leakage from late fields, growing counts and identifiers, and from near-duplicates and future tickets in a random split.
* Error analysis on validation mistakes, with a ceiling for each fix.
* Label agreement on the rare class, and Cohen's kappa.
* Drift in the mix, which moves precision, and drift in meaning, which silently lowers recall; and the weekly lines that watch for both.

The next lesson leaves learning from data for a while and starts searching for plans.



Run the cell below any time to see how far you are.

In [ ]:
progress()

<font size="2" color="#8a84b3">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</font>